# 📻 Radio QC — tìm quảng cáo trong file phát sóng

**Cách dùng:**
1. Bỏ file phát sóng vào thư mục **RadioQC/2_FilePhatSong** trong Google Drive của bạn (file quảng cáo gốc — spot —
   nếu có, vào **RadioQC/1_ThuVienQC**). Lần đầu chưa có thư mục thì cứ chạy: notebook tự tạo rồi nhắc bạn.
2. Bấm **Runtime → Run all** (Ctrl+F9). Colab báo notebook *không do Google viết* → bấm **Run anyway** (Vẫn chạy);
   Colab xin quyền vào Google Drive → chọn tài khoản của bạn → **Cho phép**.
3. Đợi đến khi xong: **file Excel kết quả tự tải về máy** và có một bản trong **RadioQC/3_KetQua**; bảng kết quả hiện
   ngay bên dưới, bấm ▶ để nghe từng quảng cáo. Lần sau notebook chỉ xử lý file mới.

Có file spot: notebook dò **chính xác** từng lần phát của spot đó. Không có: **AI tự nhận diện** quảng cáo từ lời nói
(nên nghe lại để xác nhận), kèm bản chép lời trong sheet *Chép lời*. File 3 giờ có thể mất 10–20 phút.

**Kho spot tự lớn dần:** quảng cáo mới AI tìm được được lưu làm spot trong **RadioQC/1_ThuVienQC/Chờ duyệt**; lần sau
notebook dò chúng bằng vân tay (chính xác, bắt được cả quảng cáo toàn nhạc) nhưng ghi *Cần nghe lại* cho tới khi bạn
duyệt: nghe file, đúng thì đổi tên theo nhãn hàng rồi kéo ra **1_ThuVienQC**, sai thì xoá. Tắt: bỏ tick `TU_HOC_SPOT`.

Không muốn dùng Drive: ở ô ⚙️ Tuỳ chọn, chọn `NGUON_FILE` = *Tải lên từ máy* rồi bấm Choose Files (chậm với file lớn).
Tên file phát sóng nên có ngày giờ bắt đầu, ví dụ `CAO DIEM CHIEU 25.9.2026 16H30.mp3`, để báo cáo ghi giờ thật.
Hướng dẫn đầy đủ: https://github.com/nguyenthanhhai-rf/radio-qc-colab


In [ ]:
#@title ⚙️ Tuỳ chọn (không cần sửa)
NGUON_FILE = "Google Drive"  #@param ["Google Drive", "Tải lên từ máy"]
THU_MUC_DRIVE = "RadioQC"  #@param {type:"string"}
CHAY_LAI_TAT_CA = False  #@param {type:"boolean"}
TU_HOC_SPOT = True  #@param {type:"boolean"}


In [ ]:
#@title 1️⃣ Lấy file: từ Google Drive, hoặc bấm "Choose Files" nếu chọn tải lên từ máy
import pathlib
import sys
_FILES = {'radioqc/__init__.py': '"""Radio QC: find radio advertisements in long broadcast recordings."""\n\n__version__ = "0.1.0"\n', 'radioqc/audio.py': '"""Decode audio files into mono float32 arrays (PyAV / FFmpeg).\n\nPyAV is imported inside the functions so the notebook can show the file picker before\ninstalling anything.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\n\nAUDIO_EXTENSIONS = {".mp3", ".mp2", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma", ".mp4"}\n\n\ndef is_audio_file(path: Path) -> bool:\n    return path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS and not path.name.startswith(".")\n\n\ndef load_audio(path: str | Path, sr: int, start_s: float = 0.0,\n               duration_s: float | None = None) -> np.ndarray:\n    """Decode `path` to mono float32 in [-1, 1] at `sr` Hz.\n\n    A window (`start_s`, `duration_s`) relies on container seeking, which is only\n    approximate for some formats: use windows for listening, never for measurements.\n    """\n    import av\n\n    chunks: list[np.ndarray] = []\n    have, skip = 0, 0\n    need = None if duration_s is None else max(0, int(round(duration_s * sr)))\n    with av.open(str(path)) as container:\n        if not container.streams.audio:\n            raise ValueError(f"không có luồng âm thanh trong {Path(path).name}")\n        stream = container.streams.audio[0]\n        resampler = av.AudioResampler(format="s16", layout="mono", rate=sr)\n        origin: float | None = 0.0\n        if start_s > 0:\n            container.seek(int(max(0.0, start_s - 1.0) * 1_000_000))\n            origin = None\n        finished = True\n        for frame in container.decode(stream):\n            if origin is None:\n                origin = frame.time if frame.time is not None else max(0.0, start_s - 1.0)\n                skip = max(0, int(round((start_s - origin) * sr)))\n            for out in resampler.resample(frame):\n                a = out.to_ndarray().reshape(-1)\n                chunks.append(a)\n                have += a.size\n            if need is not None and have >= skip + need:\n                finished = False\n                break\n        if finished:\n            for out in resampler.resample(None):\n                a = out.to_ndarray().reshape(-1)\n                chunks.append(a)\n                have += a.size\n    x = np.concatenate(chunks) if chunks else np.zeros(0, np.int16)\n    x = x[skip:] if need is None else x[skip:skip + need]\n    return x.astype(np.float32) / 32768.0\n\n\ndef duration_of(path: str | Path) -> float:\n    """Length in seconds from the container header; decodes the file only when the header has none."""\n    import av\n\n    with av.open(str(path)) as container:\n        if container.duration:\n            return container.duration / 1_000_000\n    return load_audio(path, 8000).size / 8000\n', 'radioqc/classify.py': '"""Ask a language model which transcript lines are ads (Gemini through google.colab.ai in Colab).\n\nThe transcript goes out in ~15-minute chunks that overlap by 2 minutes, each line numbered\nwith its position in the file. A chunk whose answer never parses is reported as not\nanalysed: it must never read as "no ads here".\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport time\nimport unicodedata\nfrom collections.abc import Callable, Collection\nfrom dataclasses import dataclass, field\nfrom difflib import SequenceMatcher\n\nfrom .report import hms\nfrom .transcript import Line\n\nLLM = Callable[[str], str]\nCHUNK_S = 900.0\nOVERLAP_S = 120.0\nRETRIES = 1                    # a malformed answer is asked again at once, this many times\nBUSY_WAITS_S = (30.0, 90.0)    # a busy AI gets this long before each retry (2026-09-28: Colab\'s Gemini said 429\n                               # "heavy load" for 5 of 12 chunks, and the instant retry was refused as well)\nWAIT_BUDGET_S = 600.0          # at most this much waiting per file: a dead AI must not hold the run for an hour\n_BUSY = re.compile(r"\\b429\\b|rate.?limit|heavy load|overload|resource.?exhausted|quota|unavailable|\\b503\\b|try again",\n                   re.IGNORECASE)\nQUOTE_MATCH = 0.8   # similarity between the words the model quoted and the start of a line\nCATEGORIES = ("QC thương mại", "Đài tự giới thiệu", "Tài trợ")\n\nPROMPT = """Bạn là chuyên viên giám sát quảng cáo trên radio Việt Nam. Dưới đây là bản chép lời tự động (có thể sai chính tả) của một đoạn phát sóng; mỗi dòng có số thứ tự và vị trí trong file.\n\nHãy tìm MỌI đoạn quảng cáo, gồm ba loại:\n- "QC thương mại": quảng cáo sản phẩm, dịch vụ, dự án bất động sản của doanh nghiệp (thường có tên nhãn hàng, số điện thoại, website, "một sản phẩm của...").\n- "Đài tự giới thiệu": đài quảng bá chương trình của chính đài ("hãy đón nghe...", "chương trình ... phát sóng vào...").\n- "Tài trợ": lời giới thiệu hoặc cảm ơn nhà tài trợ chương trình.\nKHÔNG tính: tin tức, thông tin giao thông, lời dẫn của người dẫn chương trình, trò chuyện với thính giả, bài hát, câu nhận diện đài (tên đài, tần số, khẩu hiệu của đài, không nhắc tới chương trình cụ thể nào).\n\nMỗi quảng cáo là một phần tử riêng: hai quảng cáo liền nhau của hai nhãn hàng khác nhau là hai phần tử.\nRanh giới phải chính xác:\n- "dau" là dòng đầu tiên của chính quảng cáo, "cuoi" là dòng cuối cùng của nó. KHÔNG gộp lời dẫn, tin giao thông hay lời trò chuyện của người dẫn chương trình ngay trước hoặc ngay sau quảng cáo.\n- Một quảng cáo thường dài 15–70 giây. Phần tử dài quá 90 giây gần như chắc chắn gồm nhiều quảng cáo liền nhau hoặc đã lẫn lời dẫn: hãy tách ở chỗ đổi nhãn hàng, đổi giọng hoặc đổi chủ đề.\n- "Đài tự giới thiệu" chỉ gồm lời quảng bá chương trình, không gồm nội dung của chính chương trình đó.\nTên nhãn hàng: ghi đúng nhãn hàng được nói trong quảng cáo (được sửa lỗi chính tả do máy chép lời), KHÔNG thay bằng một nhãn hàng khác, kể cả hãng nổi tiếng hơn cùng ngành.\nDòng có dấu 🔁 là âm thanh lặp lại ở chỗ khác trong file (quảng cáo thường phát nhiều lần); đó chỉ là gợi ý.\n\nChỉ trả lời bằng JSON, không giải thích, theo dạng:\n[{"loi_dau": "<3–6 chữ đầu của dòng đầu, chép nguyên văn>", "dau": <số dòng đầu>, "loi_cuoi": "<3–6 chữ đầu của dòng cuối, chép nguyên văn>", "cuoi": <số dòng cuối>, "nhan_hang": "...", "san_pham": "...", "loai": "QC thương mại"}]\n"loai" là một trong: "QC thương mại", "Đài tự giới thiệu", "Tài trợ". Không có quảng cáo thì trả về [].\n\nBản chép lời:\n<<LINES>>\n"""\n\n\n@dataclass(frozen=True)\nclass AdSpan:\n    first: int      # index of the first transcript line (whole file)\n    last: int       # index of the last line, inclusive\n    brand: str\n    product: str\n    category: str\n\n\n@dataclass\nclass Classification:\n    spans: list[AdSpan]\n    failed: list[tuple[float, float]] = field(default_factory=list)   # (start_s, end_s) not analysed\n\n\ndef _category(raw: object) -> str:\n    text = str(raw).lower()\n    if "giới thiệu" in text or "quảng bá" in text:\n        return "Đài tự giới thiệu"\n    if "tài trợ" in text:\n        return "Tài trợ"\n    return "QC thương mại"\n\n\ndef _plain(text: str) -> str:\n    """Lower case without accents or punctuation: models often miscopy Vietnamese diacritics."""\n    text = unicodedata.normalize("NFD", text.lower().replace("đ", "d"))\n    text = "".join(c for c in text if not unicodedata.combining(c))\n    return " ".join(re.sub(r"[^\\w\\s]", " ", text).split())\n\n\ndef _says(line: str, quote: str) -> bool:\n    q, t = _plain(quote), _plain(line)\n    return bool(q) and (q in t or SequenceMatcher(None, q, t[:len(q)]).ratio() >= QUOTE_MATCH)\n\n\ndef _anchor(idx: int, quote: str, lines: list[Line], valid: range) -> int | None:\n    """Where the quoted line really is: `idx` when its text matches, else the nearest line that does."""\n    if idx in valid and _says(lines[idx].text, quote):\n        return idx\n    hits = [i for i in valid if _says(lines[i].text, quote)]\n    return min(hits, key=lambda i: abs(i - idx)) if hits else None\n\n\ndef _anchored(first: int, last: int, quote_first: str, quote_last: str, lines: list[Line],\n              valid: range) -> tuple[int, int]:\n    """Line numbers checked against the words the model quoted from those lines.\n\n    On 25/9 Gemini named the right ads in the right order but numbered them ~46 lines too early, which\n    put "KDV" on traffic news: a silent error, since the report still looked plausible. A quote found\n    elsewhere moves the span; one end found moves both by the same amount; nothing found keeps the numbers.\n    """\n    f = _anchor(first, quote_first, lines, valid) if quote_first.strip() else None\n    t = _anchor(last, quote_last, lines, valid) if quote_last.strip() else None\n    if f is None and t is None:\n        return first, last\n    return (f if f is not None else first + (t - last)), (t if t is not None else last + (f - first))\n\n\ndef parse_reply(reply: str, valid: range, lines: list[Line] | None = None) -> list[AdSpan]:\n    """Spans from the model\'s JSON answer; raises ValueError when there is no JSON list at all.\n\n    Models sometimes answer, comment, then send a corrected list: the last complete list of objects wins\n    (a later non-empty one over an empty one, so a stray "[]" in a note cannot silently mean "no ads").\n    With `lines`, each span\'s numbers are checked against the words the model quoted (`_anchored`).\n    """\n    decoder, lists, pos = json.JSONDecoder(), [], reply.find("[")\n    while pos != -1:\n        try:\n            data, end = decoder.raw_decode(reply, pos)\n        except ValueError:\n            end = pos + 1\n        else:\n            if isinstance(data, list) and all(isinstance(x, dict) for x in data):\n                lists.append(data)\n        pos = reply.find("[", end)\n    if not lists:\n        raise ValueError("câu trả lời không có danh sách JSON")\n    data = next((x for x in reversed(lists) if x), lists[-1])\n    spans = []\n    for item in data:\n        if not isinstance(item, dict):\n            continue\n        try:\n            first, last = int(item["dau"]), int(item["cuoi"])\n        except (KeyError, TypeError, ValueError):\n            continue\n        if lines is not None:\n            first, last = _anchored(first, last, str(item.get("loi_dau") or ""), str(item.get("loi_cuoi") or ""),\n                                    lines, valid)\n        if first > last or first not in valid or last not in valid:\n            continue\n        spans.append(AdSpan(first, last, str(item.get("nhan_hang") or "").strip(),\n                            str(item.get("san_pham") or "").strip(), _category(item.get("loai", ""))))\n    return spans\n\n\ndef _chunks(lines: list[Line]) -> list[tuple[int, int]]:\n    """(first, last) line indices of each ~15-minute chunk, overlapping by OVERLAP_S."""\n    out: list[tuple[int, int]] = []\n    if not lines:\n        return out\n    t0, end = lines[0].start, lines[-1].end\n    while True:\n        idx = [i for i, ln in enumerate(lines) if t0 <= ln.start < t0 + CHUNK_S + OVERLAP_S]\n        if idx and (not out or idx[-1] > out[-1][1]):\n            out.append((idx[0], idx[-1]))\n        t0 += CHUNK_S\n        if t0 >= end:\n            return out\n\n\ndef _merge(spans: list[AdSpan]) -> list[AdSpan]:\n    """Join duplicates from overlapping chunks: same ad = same brand, or mostly the same lines."""\n    merged: list[AdSpan] = []\n    for s in sorted(spans, key=lambda s: (s.first, s.last)):\n        if merged:\n            m = merged[-1]\n            overlap = min(m.last, s.last) - max(m.first, s.first) + 1\n            shorter = min(m.last - m.first, s.last - s.first) + 1\n            same_brand = m.brand.lower() == s.brand.lower() and overlap > 0\n            if same_brand or overlap >= 0.5 * shorter:\n                keep = m if (m.last - m.first) >= (s.last - s.first) else s\n                merged[-1] = AdSpan(min(m.first, s.first), max(m.last, s.last), keep.brand or m.brand or s.brand,\n                                    keep.product or m.product or s.product, keep.category)\n                continue\n        merged.append(s)\n    return merged\n\n\ndef classify(lines: list[Line], llm: LLM, repeated: Collection[int] = (),\n             log: Callable[[str], None] = print, sleep: Callable[[float], None] = time.sleep) -> Classification:\n    """Ad spans for the whole transcript; `repeated` holds indices of lines inside repeated audio.\n\n    A busy AI (429, overload) is given BUSY_WAITS_S before each retry, within WAIT_BUDGET_S for the whole\n    file; a malformed answer is asked again at once. A chunk that still fails is reported as not analysed.\n    """\n    spans: list[AdSpan] = []\n    failed: list[tuple[float, float]] = []\n    budget = WAIT_BUDGET_S\n    for a, b in _chunks(lines):\n        where = f"{hms(lines[a].start)}–{hms(lines[b].end)}"\n        body = "\\n".join(f"{i}. [{hms(lines[i].start)}] {\'🔁 \' if i in repeated else \'\'}{lines[i].text}"\n                         for i in range(a, b + 1))\n        prompt = PROMPT.replace("<<LINES>>", body)\n        waits, retries = list(BUSY_WAITS_S), RETRIES\n        while True:\n            try:\n                spans += parse_reply(llm(prompt), valid=range(a, b + 1), lines=lines)\n                break\n            except Exception as e:  # noqa: BLE001 - busy, quota, network, malformed answer: retry, then report\n                busy = bool(_BUSY.search(str(e)))\n                if busy and waits and waits[0] <= budget:\n                    wait = waits.pop(0)\n                    budget -= wait\n                    log(f"⏳ Đoạn {where}: AI đang quá tải, đợi {wait:.0f} giây rồi thử lại…")\n                    sleep(wait)\n                elif not busy and retries > 0:\n                    retries -= 1\n                    log(f"⚠️ Đoạn {where}: AI chưa trả lời được ({e}), thử lại…")\n                else:\n                    log(f"⚠️ Đoạn {where}: AI chưa trả lời được ({e}), bỏ qua đoạn này.")\n                    failed.append((lines[a].start, lines[b].end))\n                    break\n    return Classification(_merge(spans), failed)\n\n\ndef colab_gemini(model_name: str | None = None) -> LLM:\n    """Gemini through Colab\'s built-in, key-free `google.colab.ai` (the user\'s own free quota)."""\n    from google.colab import ai\n\n    def ask(prompt: str) -> str:\n        return ai.generate_text(prompt, model_name=model_name) if model_name else ai.generate_text(prompt)\n    return ask\n', 'radioqc/clips.py': '"""Small MP3 clips of detected ads, played right inside the notebook\'s results table."""\nfrom __future__ import annotations\n\nimport base64\nimport io\n\nimport numpy as np\n\nCLIP_BITRATE = 24_000   # speech-quality; a 30 s clip is ~100 kB\n\n\ndef mp3_clip(audio: np.ndarray, sr: int, start_s: float, end_s: float, pad_s: float = 1.0,\n             bitrate: int = CLIP_BITRATE) -> bytes:\n    """MP3 bytes of `audio[start_s - pad_s : end_s + pad_s]`."""\n    import av\n\n    a, b = max(0, int((start_s - pad_s) * sr)), min(audio.size, int((end_s + pad_s) * sr))\n    pcm = (np.clip(audio[a:b], -1, 1) * 32767).astype(np.int16).reshape(1, -1)\n    buf = io.BytesIO()\n    with av.open(buf, "w", format="mp3") as out:\n        stream = out.add_stream("libmp3lame", rate=sr, layout="mono")\n        stream.bit_rate = bitrate\n        frame = av.AudioFrame.from_ndarray(pcm, format="s16", layout="mono")\n        frame.sample_rate = sr\n        for packet in stream.encode(frame):\n            out.mux(packet)\n        for packet in stream.encode(None):\n            out.mux(packet)\n    return buf.getvalue()\n\n\ndef audio_tag(mp3: bytes) -> str:\n    data = base64.b64encode(mp3).decode("ascii")\n    return f\'<audio controls preload="none" style="height:32px" src="data:audio/mpeg;base64,{data}"></audio>\'\n', 'radioqc/discover.py': '"""Merge the three sources of evidence into one list of ads for the report.\n\n- A: airings of spots the user supplied (exact, wins when it overlaps anything else),\n- B: stretches the language model marked in the transcript,\n- repeats: audio heard more than once, used to sharpen B\'s boundaries and as extra evidence.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nfrom .classify import AdSpan\nfrom .matcher import Airing\nfrom .repeats import Repeat\nfrom .transcript import Line\n\nSNAP_S = 6.0          # a repeat within this distance of both ends replaces the transcript boundaries\nGROW_MAX_S = 90.0     # ...or of one end, holding the whole span, and no longer than one ad (spec §6)\nJOIN_GAP_S = 3.0      # two parts of the same brand this close are one ad\nSHORT_S = 5.0\n\n\n@dataclass\nclass Detection:\n    start_s: float\n    end_s: float\n    brand: str            # brand named by the AI, or the spot\'s name (A)\n    product: str\n    category: str\n    text: str             # what was said during the ad\n    evidence: list[str] = field(default_factory=list)\n    review: str = ""      # why a person should listen, "" when not needed\n    source: str = "B"     # "A" (known spot) or "B" (found by AI)\n\n    @property\n    def duration_s(self) -> float:\n        return self.end_s - self.start_s\n\n\ndef _overlap(a0: float, a1: float, b0: float, b1: float) -> float:\n    return max(0.0, min(a1, b1) - max(a0, b0))\n\n\ndef _fits(d: Detection, a: float, b: float, others: list[Detection]) -> bool:\n    """Should repeat occurrence [a, b] replace the AI\'s boundaries of `d`?\n\n    Both ends close: yes. One end close and `d` inside it: yes too, because a jingle opening often gives\n    Whisper no words (25/9: Libera began 34 s late) - unless [a, b] is longer than one ad or would swallow\n    another detection, which means it is several ads aired back to back.\n    """\n    near_start, near_end = abs(a - d.start_s) <= SNAP_S, abs(b - d.end_s) <= SNAP_S\n    if near_start and near_end:\n        return True\n    return ((near_start or near_end) and a - SNAP_S <= d.start_s and d.end_s <= b + SNAP_S and b - a <= GROW_MAX_S\n            and not any(o is not d and _overlap(o.start_s, o.end_s, a, b) > SNAP_S for o in others))\n\n\ndef _words(lines: list[Line], start: float, end: float) -> str:\n    return " ".join(ln.text for ln in lines if _overlap(ln.start, ln.end, start, end) > 0)\n\n\ndef detect(lines: list[Line], spans: list[AdSpan], repeats: list[Repeat], airings: list[Airing]) -> list[Detection]:\n    found = [Detection(a.start_s, a.end_s, a.name, "", "QC thương mại", _words(lines, a.start_s, a.end_s),\n                       [f"Khớp spot trong kho ({100 * a.coverage:.0f}%)"],\n                       "" if a.status == "Đủ" else a.status, "A") for a in airings]\n    ai: list[Detection] = []\n    for s in sorted(spans, key=lambda s: s.first):\n        start, end = lines[s.first].start, lines[s.last].end\n        if ai and ai[-1].brand.lower() == s.brand.lower() and s.brand and start - ai[-1].end_s <= JOIN_GAP_S:\n            ai[-1].end_s = end\n            continue\n        ai.append(Detection(start, end, s.brand, s.product, s.category, "", ["AI đọc lời"]))\n    for d in ai:\n        for rep in repeats:\n            hit = next(((a, b) for a, b in rep.occurrences if _fits(d, a, b, ai)), None)\n            if hit:\n                d.start_s, d.end_s = hit\n                d.evidence.append(f"Lặp lại {rep.count} lần trong file")\n                break\n        d.text = _words(lines, d.start_s, d.end_s)\n        known = next((k for k in found if k.source == "A"\n                      and _overlap(k.start_s, k.end_s, d.start_s, d.end_s) > 0.5 * min(k.duration_s, d.duration_s)),\n                     None)\n        if known:\n            known.evidence.append(f"AI cũng nhận ra: {d.brand or \'quảng cáo\'}")\n            continue\n        reasons = []\n        if not d.brand and d.category == "QC thương mại":\n            reasons.append("chưa rõ nhãn hàng")\n        if d.duration_s < SHORT_S:\n            reasons.append("rất ngắn")\n        d.review = ", ".join(reasons)\n        found.append(d)\n    return sorted(found, key=lambda d: d.start_s)\n', 'radioqc/fingerprint.py': '"""Landmark fingerprints: pairs of spectral peaks, robust to broadcast processing.\n\nEvery parameter below changes the hashes. Bump FP_VERSION whenever one changes so\nthat cached library fingerprints are rebuilt.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\nfrom numpy.lib.stride_tricks import sliding_window_view\nfrom scipy.ndimage import maximum_filter\n\nFP_VERSION = 1\nSR = 8000                 # fingerprint sample rate (Hz)\nN_FFT = 512               # 64 ms window\nHOP = 256                 # 32 ms hop\nFPS = SR / HOP            # frames per second (31.25)\nF_LO, F_HI = 6, 240       # bins used for peaks (~94 Hz - 3.75 kHz)\nPEAK_SIZE = 15            # local-maximum neighbourhood, in bins and frames\nPEAK_ABOVE_MEDIAN = 2.0   # log-magnitude above the frame median (~17 dB)\nLOUD_FLOOR = float(np.log(0.05))  # quieter frames carry no peaks\nFAN_OUT = 5               # targets paired with each anchor peak\nDT_MAX = 32               # target at most 32 frames (~1 s) after the anchor\nDF_MAX = 32               # target within +/-32 bins of the anchor\nPAIR_SCAN = 80            # later peaks scanned per anchor when pairing\nBLOCK = 8192              # frames per spectrogram block (~4.4 min), bounds memory\n\n\n@dataclass\nclass Fingerprint:\n    hashes: np.ndarray    # uint32, one per landmark\n    times: np.ndarray     # int32 anchor frame of each landmark\n    n_frames: int\n    duration_s: float\n\n\ndef _peaks(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, int]:\n    """Spectral peaks as (frame, bin) arrays sorted by frame, then bin."""\n    n_frames = 1 + (x.size - N_FFT) // HOP if x.size >= N_FFT else 0\n    if n_frames == 0:\n        empty = np.zeros(0, np.int64)\n        return empty, empty, 0\n    frames = sliding_window_view(x, N_FFT)[::HOP]\n    win = np.hanning(N_FFT).astype(np.float32)\n    pad = PEAK_SIZE // 2\n    ts, fs = [], []\n    for b0 in range(0, n_frames, BLOCK):\n        b1 = min(n_frames, b0 + BLOCK)\n        c0, c1 = max(0, b0 - pad), min(n_frames, b1 + pad)\n        spec = np.abs(np.fft.rfft(frames[c0:c1] * win, axis=1)).T\n        S = np.log(spec + 1e-6).astype(np.float32)\n        band = S[F_LO:F_HI]\n        is_max = band == maximum_filter(band, size=PEAK_SIZE, mode="constant", cval=-np.inf)\n        strong = band > np.median(band, axis=0, keepdims=True) + PEAK_ABOVE_MEDIAN\n        loud = S.max(axis=0) > LOUD_FLOOR\n        f, t = np.nonzero(is_max & strong & loud[None, :])\n        t = t + c0\n        keep = (t >= b0) & (t < b1)\n        ts.append(t[keep])\n        fs.append(f[keep] + F_LO)\n    t = np.concatenate(ts).astype(np.int64)\n    f = np.concatenate(fs).astype(np.int64)\n    order = np.lexsort((f, t))\n    return t[order], f[order], n_frames\n\n\ndef _pairs(t: np.ndarray, f: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Pair each anchor peak with up to FAN_OUT later peaks in its target zone."""\n    n = t.size\n    used = np.zeros(n, np.int32)\n    hashes, times = [], []\n    for s in range(1, min(PAIR_SCAN, n - 1) + 1):\n        i = np.arange(n - s)\n        j = i + s\n        dt = t[j] - t[i]\n        if dt.min() > DT_MAX:\n            break\n        df = f[j] - f[i]\n        ok = (dt >= 1) & (dt <= DT_MAX) & (np.abs(df) <= DF_MAX) & (used[i] < FAN_OUT)\n        i = i[ok]\n        used[i] += 1\n        hashes.append((f[i] << 13) | ((df[ok] + DF_MAX) << 6) | dt[ok])\n        times.append(t[i])\n    if not hashes:\n        return np.zeros(0, np.uint32), np.zeros(0, np.int32)\n    return np.concatenate(hashes).astype(np.uint32), np.concatenate(times).astype(np.int32)\n\n\ndef fingerprint(x: np.ndarray) -> Fingerprint:\n    """Fingerprint mono audio sampled at SR."""\n    x = np.ascontiguousarray(x, dtype=np.float32)\n    t, f, n_frames = _peaks(x)\n    hashes, times = _pairs(t, f)\n    return Fingerprint(hashes, times, n_frames, x.size / SR)\n', 'radioqc/learn.py': '"""The spot library grows by itself: ads the AI found become spot files, matched by fingerprint next time.\n\nNew spots go to 1_ThuVienQC/Chờ duyệt. They are used at once, but what they find is marked for a listen\nuntil the user approves them by moving the file out (renamed to the brand as they like); a wrong one is\ndeleted. Only clear commercial ads are learned, one file per distinct audio (spec §6b).\n"""\nfrom __future__ import annotations\n\nimport re\nimport wave\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .discover import Detection\nfrom .fingerprint import FPS, SR, Fingerprint, fingerprint\nfrom .library import LibraryIndex\nfrom .matcher import find_airings\nfrom .naming import BroadcastName\nfrom .repeats import Repeat\nfrom .report import hms\n\nPENDING_SUBDIR = "Chờ duyệt"\nPENDING_REVIEW = "spot tự học chưa duyệt"\nLEARN_CATEGORY = "QC thương mại"\nLEARN_MIN_S, LEARN_MAX_S = 10.0, 90.0\nTRIM_S = 1.0             # transcript edges are ~1 s off: cut inside them so no neighbouring audio gets in\n_UNKNOWN = {"", "chưa rõ", "không rõ", "unknown"}\n\n\ndef _snapped(d: Detection, repeats: list[Repeat]) -> Repeat | None:\n    """The repeat whose occurrence gave `d` its edges (discover.detect copies them exactly)."""\n    return next((r for r in repeats for a, b in r.occurrences\n                 if abs(a - d.start_s) < 1e-6 and abs(b - d.end_s) < 1e-6), None)\n\n\ndef candidates(detections: list[Detection], repeats: list[Repeat]) -> list[tuple[Detection, float, float]]:\n    """(detection, start_s, end_s) worth keeping as a spot: commercial ads the AI found with a brand and no\n    doubt flagged, 10-90 s once cut. Edges from a repeat are exact and the repeat is learned once; edges\n    from the transcript are pulled in by TRIM_S."""\n    out: list[tuple[Detection, float, float]] = []\n    seen: list[Repeat] = []\n    for d in detections:\n        if d.source != "B" or d.category != LEARN_CATEGORY or d.review or d.brand.strip().lower() in _UNKNOWN:\n            continue\n        rep = _snapped(d, repeats)\n        if rep is not None:\n            if any(rep is r for r in seen):\n                continue\n            seen.append(rep)\n            a, b = d.start_s, d.end_s\n        else:\n            a, b = d.start_s + TRIM_S, d.end_s - TRIM_S\n        if LEARN_MIN_S <= b - a <= LEARN_MAX_S:\n            out.append((d, a, b))\n    return out\n\n\ndef _stem(brand: str, name: BroadcastName, at_s: float) -> str:\n    brand = re.sub(r"\\s+", " ", re.sub(r\'[\\\\/:*?"<>|]+\', " - ", brand)).strip(" -.") or "Quảng cáo"\n    if name.start is None:\n        return f"{brand} +{hms(at_s).replace(\':\', \'h\', 1).replace(\':\', \'m\')}"\n    t = name.start + timedelta(seconds=int(at_s))\n    return f"{brand} {t:%d.%m} {t:%H}h{t:%M}"\n\n\ndef cut_start(t: float) -> float:\n    """Where a spot starting near `t` is cut: half a fingerprint frame off the recording\'s frame grid.\n\n    That is the hardest alignment for the matcher, the one a later airing can meet at worst, so `verified`\n    tests every spot against it. Cut on the grid, a spot always matches its own source and the test says\n    nothing (on 25/9 two such cuts matched only 71-82 % once off the grid).\n    """\n    return (np.floor(t * FPS) + 0.5) / FPS\n\n\ndef verified(audio: np.ndarray, fp: Fingerprint, cands: list[tuple[Detection, float, float]]\n             ) -> list[tuple[Detection, float, float]]:\n    """The candidates whose cut of `audio` (at the fingerprint rate) matches its own airing in full ("Đủ").\n\n    A learned spot must never report a cut airing that did not happen, so a spot that only matches part of\n    itself once off the frame grid (`cut_start`) is not learned: on 25/9 that dropped 4 of 36 candidates,\n    which would have reported "Phát thiếu ~4 s". The same ad cut at two airings (when no repeat linked\n    them) is learned once: the matcher lists the other cut under `also_matched` where both fit.\n    """\n    if not cands:\n        return []\n    index = LibraryIndex.from_fingerprints([(str(k), str(k), fingerprint(audio[round(cut_start(a) * SR):round(b * SR)]))\n                                            for k, (_, a, b) in enumerate(cands)])\n    airings = find_airings(index, fp)\n    whole = {x.spot_id for x in airings\n             if x.status == "Đủ" and abs(x.offset_s - cut_start(cands[x.spot_id][1])) <= 1.0}\n    copies: dict[int, set[int]] = {}\n    for x in airings:\n        group = {x.spot_id, *(int(n) for n in x.also_matched)}\n        for k in group:\n            copies.setdefault(k, set()).update(group - {k})\n    kept: list[int] = []\n    for k in sorted(whole):\n        if not copies.get(k, set()) & set(kept):\n            kept.append(k)\n    return [cands[k] for k in kept]\n\n\ndef save_spots(audio: np.ndarray, sr: int, cands: list[tuple[Detection, float, float]], name: BroadcastName,\n               folder: Path) -> list[Path]:\n    """Write each candidate as \'<brand> <dd.mm> <HHhMM>.wav\' in `folder`, never overwriting a file.\n\n    WAV, not MP3: lossless keeps the landmarks `verified` counted (8 kHz MP3 lost 18-29 % on 25/9), and a\n    WAV still plays in Google Drive for the person who approves it.\n    """\n    folder.mkdir(parents=True, exist_ok=True)\n    saved = []\n    for d, a, b in cands:\n        stem = _stem(d.brand, name, a)\n        path, k = folder / f"{stem}.wav", 1\n        while path.exists():\n            k += 1\n            path = folder / f"{stem} ({k}).wav"\n        pcm = (np.clip(audio[round(cut_start(a) * sr):round(b * sr)], -1, 1) * 32767).astype("<i2")\n        with wave.open(str(path), "wb") as w:\n            w.setnchannels(1)\n            w.setsampwidth(2)\n            w.setframerate(sr)\n            w.writeframes(pcm.tobytes())\n        saved.append(path)\n    return saved\n', 'radioqc/library.py': '"""Spot library: fingerprints of known ads, indexed by hash and cached on disk."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections.abc import Callable\n\nimport numpy as np\n\nfrom .audio import is_audio_file, load_audio\nfrom .fingerprint import FP_VERSION, FPS, SR, Fingerprint, fingerprint\n\nMIN_SPOT_S = 5.0  # shorter clips (sound logos) cannot be matched reliably\n\n\n@dataclass\nclass Spot:\n    name: str\n    file: str\n    duration_s: float\n    hashes: np.ndarray        # sorted\n    times: np.ndarray         # aligned with hashes\n    content_secs: np.ndarray  # sorted whole seconds that contain landmarks\n\n    @property\n    def n_landmarks(self) -> int:\n        return int(self.hashes.size)\n\n\n@dataclass\nclass LibraryIndex:\n    spots: list[Spot]\n    hashes: np.ndarray        # every landmark of every spot, sorted by hash\n    spot_ids: np.ndarray\n    times: np.ndarray\n    problems: list[tuple[str, str]] = field(default_factory=list)\n\n    @classmethod\n    def from_fingerprints(cls, items: list[tuple[str, str, Fingerprint]],\n                          problems: list[tuple[str, str]] | None = None) -> LibraryIndex:\n        """items: (name, file, fingerprint) for each spot."""\n        spots = []\n        for name, file, fp in items:\n            o = np.argsort(fp.hashes, kind="stable")\n            secs = np.unique((fp.times / FPS).astype(np.int64))\n            spots.append(Spot(name, file, fp.duration_s, fp.hashes[o], fp.times[o], secs))\n        if spots:\n            h = np.concatenate([s.hashes for s in spots])\n            ids = np.concatenate([np.full(s.n_landmarks, k, np.int32) for k, s in enumerate(spots)])\n            t = np.concatenate([s.times for s in spots])\n            o = np.argsort(h, kind="stable")\n            h, ids, t = h[o], ids[o], t[o]\n        else:\n            h, ids, t = np.zeros(0, np.uint32), np.zeros(0, np.int32), np.zeros(0, np.int32)\n        return cls(spots, h, ids, t, list(problems or []))\n\n\ndef _meta(rel: str, st: os.stat_result) -> dict:\n    return {"file": rel, "size": st.st_size, "mtime_ns": st.st_mtime_ns, "fp_version": FP_VERSION}\n\n\ndef _load_cached(path: Path, meta: dict) -> Fingerprint | None:\n    try:\n        with np.load(path, allow_pickle=False) as z:\n            if json.loads(str(z["meta"])) != meta:\n                return None\n            return Fingerprint(z["hashes"], z["times"], int(z["n_frames"]), float(z["duration_s"]))\n    except (OSError, KeyError, ValueError):\n        return None\n\n\ndef _save_cached(path: Path, meta: dict, fp: Fingerprint) -> None:\n    tmp = path.with_name(path.name + ".tmp.npz")\n    np.savez(tmp, hashes=fp.hashes, times=fp.times, n_frames=fp.n_frames,\n             duration_s=fp.duration_s, meta=json.dumps(meta, sort_keys=True))\n    os.replace(tmp, path)\n\n\ndef build_library(folder: Path, cache_dir: Path, log: Callable[[str], None] = print) -> LibraryIndex:\n    """Fingerprint every audio file under `folder` (re-using the cache) and index them."""\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    files = sorted((p for p in folder.rglob("*") if is_audio_file(p)),\n                   key=lambda p: p.relative_to(folder).as_posix().lower())\n    items, problems = [], []\n    for p in files:\n        rel = p.relative_to(folder).as_posix()\n        meta = _meta(rel, p.stat())\n        cpath = cache_dir / (hashlib.sha1(rel.encode("utf-8")).hexdigest()[:16] + ".npz")\n        fp = _load_cached(cpath, meta)\n        if fp is None:\n            try:\n                fp = fingerprint(load_audio(p, SR))\n            except Exception as e:  # noqa: BLE001 - list every unreadable file, keep going\n                problems.append((rel, f"không đọc được file ({e})"))\n                continue\n            _save_cached(cpath, meta, fp)\n            log(f"  + vân tay: {rel}")\n        if fp.duration_s < MIN_SPOT_S:\n            problems.append((rel, f"quá ngắn ({fp.duration_s:.1f} s < {MIN_SPOT_S:.0f} s) để dò tin cậy"))\n        elif fp.hashes.size == 0:\n            problems.append((rel, "không có nội dung để lấy vân tay (im lặng?)"))\n        else:\n            items.append((p.stem, rel, fp))\n    stems = [n for n, _, _ in items]\n    items = [(rel.rsplit(".", 1)[0] if stems.count(n) > 1 else n, rel, fp) for n, rel, fp in items]\n    return LibraryIndex.from_fingerprints(items, problems)\n', 'radioqc/matcher.py': '"""Locate airings of library spots inside a broadcast fingerprint.\n\nAn airing shows up as many landmark pairs agreeing on one (spot, time offset).\nThresholds are fixed by tests (synthetic + the real 25/9 broadcast), not by feel.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom .fingerprint import DT_MAX, FPS, Fingerprint\nfrom .library import LibraryIndex, Spot\n\nMIN_PAIRS_ABS = 10        # consistent landmark pairs needed to consider an airing\n                          # (real 25/9 broadcast: random offsets reach 2-6 pairs; a worst-aligned\n                          # re-airing kept only 8.5% of pairs, so 20 would miss 5 s spots)\nMIN_PAIRS_REL = 0.02      # ... or this share of the spot\'s landmarks, if larger\nMIN_COVERAGE = 0.4        # share of the spot\'s content seconds that must be found\nMIN_MATCHED_S = 4         # distinct matched seconds\nPARTIAL_GAP_S = 2.0       # missing head/tail longer than this => partial airing\nMID_GAP_S = 4             # unmatched content seconds in a row inside the airing => doubtful\nFULL_INNER_COVERAGE = 0.8  # below this share inside the matched span => doubtful\nTIE_S = 3.0               # matched durations this close are decided by coverage\nQUERY_BLOCK = 200_000     # query landmarks per block when collecting candidates\n\n\n@dataclass\nclass Airing:\n    spot_id: int\n    name: str\n    offset_s: float           # broadcast time aligned with the spot\'s first sample\n    start_s: float\n    end_s: float\n    coverage: float           # share of the spot\'s content seconds found\n    matched_s: int            # distinct spot seconds found\n    missing_head_s: float\n    missing_tail_s: float\n    cut_by_file_start: bool   # the missing head lies before the recording started\n    cut_by_file_end: bool     # the missing tail lies after the recording ended\n    n_pairs: int\n    inner_coverage: float = 1.0  # share of content seconds found between the first and last match\n    missing_mid_s: int = 0       # longest run of unmatched content seconds inside the airing\n    also_matched: list[str] = field(default_factory=list)\n\n    @property\n    def aired_s(self) -> float:\n        return self.end_s - self.start_s\n\n    @property\n    def status(self) -> str:\n        """\'Đủ\', or why not: doubts about identity outrank a short airing."""\n        doubt, short, cut = [], [], []\n        if self.missing_mid_s > MID_GAP_S:\n            doubt.append(f"không khớp đoạn giữa ~{self.missing_mid_s} s (có thể là spot khác cùng nhạc hiệu)")\n        if self.inner_coverage < FULL_INNER_COVERAGE:\n            doubt.append(f"chỉ khớp {100 * self.inner_coverage:.0f}% phần đã phát")\n        if self.missing_head_s > PARTIAL_GAP_S:\n            if self.cut_by_file_start:\n                cut.append("đầu nằm ngoài file ghi")\n            else:\n                short.append(f"thiếu đầu ~{self.missing_head_s:.0f} s")\n        if self.missing_tail_s > PARTIAL_GAP_S:\n            if self.cut_by_file_end:\n                cut.append("cuối nằm ngoài file ghi")\n            else:\n                short.append(f"thiếu cuối ~{self.missing_tail_s:.0f} s")\n        if doubt:\n            return "Cần nghe lại: " + ", ".join(doubt + short + cut)\n        if short:\n            return "Phát thiếu: " + ", ".join(short + cut)\n        if cut:\n            return "Không kiểm được đủ: " + ", ".join(cut)\n        return "Đủ"\n\n\ndef find_airings(index: LibraryIndex, query: Fingerprint) -> list[Airing]:\n    """All airings of library spots in `query`, sorted by start time."""\n    if not index.spots or query.hashes.size == 0:\n        return []\n    order = np.argsort(query.times, kind="stable")\n    qh, qt = query.hashes[order], query.times[order].astype(np.int64)\n    found = []\n    for sid, off in _candidates(index, qh, qt, query.n_frames):\n        a = _measure(index.spots[sid], sid, off, qh, qt, query.duration_s)\n        if a is not None:\n            found.append(a)\n    return sorted(_resolve_overlaps(found), key=lambda a: a.start_s)\n\n\ndef _expand(sorted_hashes: np.ndarray, h: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Index pairs (into h, into sorted_hashes) of all equal hashes."""\n    lo = np.searchsorted(sorted_hashes, h, "left")\n    cnt = np.searchsorted(sorted_hashes, h, "right") - lo\n    total = int(cnt.sum())\n    qi = np.repeat(np.arange(h.size), cnt)\n    li = np.repeat(lo, cnt) + np.arange(total) - np.repeat(np.cumsum(cnt) - cnt, cnt)\n    return qi, li\n\n\ndef _neighbour(k: np.ndarray, c: np.ndarray, step: int) -> np.ndarray:\n    """Count stored at key k+step (0 where absent); `k` is sorted and unique."""\n    pos = np.searchsorted(k, k + step)\n    hit = pos < k.size\n    hit[hit] &= k[pos[hit]] == (k + step)[hit]\n    out = np.zeros_like(c)\n    out[hit] = c[pos[hit]]\n    return out\n\n\ndef _candidates(index: LibraryIndex, qh: np.ndarray, qt: np.ndarray,\n                n_frames: int) -> list[tuple[int, int]]:\n    """(spot_id, offset in frames) of dense offset peaks, strongest first."""\n    longest = max(int(np.ceil(s.duration_s * FPS)) for s in index.spots) + DT_MAX + 2\n    width = n_frames + 2 * longest + 2\n    keys, counts = [], []\n    for b0 in range(0, qh.size, QUERY_BLOCK):\n        qi, li = _expand(index.hashes, qh[b0:b0 + QUERY_BLOCK])\n        if qi.size == 0:\n            continue\n        off = qt[b0:b0 + QUERY_BLOCK][qi] - index.times[li].astype(np.int64)\n        key = index.spot_ids[li].astype(np.int64) * width + off + longest\n        k, c = np.unique(key, return_counts=True)\n        keys.append(k[c >= 2])\n        counts.append(c[c >= 2])\n    if not keys:\n        return []\n    k, inv = np.unique(np.concatenate(keys), return_inverse=True)\n    c = np.bincount(inv, weights=np.concatenate(counts)).astype(np.int64)\n    c3 = c + _neighbour(k, c, -1) + _neighbour(k, c, +1)   # tolerate +/-1 frame of jitter\n    sid, off = k // width, k % width - longest\n    need = np.array([max(MIN_PAIRS_ABS, MIN_PAIRS_REL * s.n_landmarks) for s in index.spots])\n    ok = np.flatnonzero(c3 >= need[sid])\n    ok = ok[np.argsort(-c3[ok], kind="stable")]\n    taken: dict[int, list[int]] = {}\n    out = []\n    for s, o in zip(sid[ok].tolist(), off[ok].tolist(), strict=True):\n        window = max(1, int(0.9 * index.spots[s].duration_s * FPS))\n        if any(abs(o - p) < window for p in taken.get(s, [])):\n            continue\n        taken.setdefault(s, []).append(o)\n        out.append((s, o))\n    return out\n\n\ndef _measure(spot: Spot, sid: int, off: int, qh: np.ndarray, qt: np.ndarray,\n             file_s: float) -> Airing | None:\n    """Coverage and boundaries of one candidate airing, or None if too weak."""\n    n_sf = int(np.ceil(spot.duration_s * FPS))\n    a, b = np.searchsorted(qt, [off - 1, off + n_sf + 2])\n    qi, li = _expand(spot.hashes, qh[a:b])\n    if qi.size == 0:\n        return None\n    tref = spot.times[li].astype(np.int64)\n    tref = tref[np.abs(qt[a:b][qi] - tref - off) <= 1]\n    secs = np.unique((tref / FPS).astype(np.int64))\n    if secs.size < MIN_MATCHED_S:\n        return None\n    content = spot.content_secs\n    coverage = secs.size / content.size\n    if coverage < MIN_COVERAGE:\n        return None\n    missing_head = float(secs[0] - content[0])\n    missing_tail = float(content[-1] - secs[-1])\n    inside = content[(content >= secs[0]) & (content <= secs[-1])]\n    offset_s = off / FPS\n    start_s = offset_s + (tref.min() / FPS if missing_head > PARTIAL_GAP_S else 0.0)\n    end_s = offset_s + (min(spot.duration_s, tref.max() / FPS + 1.0) if missing_tail > PARTIAL_GAP_S\n                        else spot.duration_s)\n    return Airing(\n        spot_id=sid, name=spot.name, offset_s=offset_s,\n        start_s=max(0.0, start_s), end_s=min(file_s, end_s),\n        coverage=float(coverage), matched_s=int(secs.size),\n        missing_head_s=missing_head, missing_tail_s=missing_tail,\n        cut_by_file_start=offset_s + secs[0] < PARTIAL_GAP_S,\n        cut_by_file_end=offset_s + secs[-1] + 1.0 > file_s - PARTIAL_GAP_S,\n        n_pairs=int(tref.size),\n        inner_coverage=float(secs.size / inside.size),\n        missing_mid_s=_longest_run(~np.isin(inside, secs)))\n\n\ndef _longest_run(flags: np.ndarray) -> int:\n    best = run = 0\n    for f in flags.tolist():\n        run = run + 1 if f else 0\n        best = max(best, run)\n    return best\n\n\ndef _resolve_overlaps(airings: list[Airing]) -> list[Airing]:\n    """Keep the spot that explains the longest stretch; list the others as \'also matched\'.\n\n    Stretches within ~TIE_S of each other are decided by coverage, so a complete 15 s\n    cut-down beats the 30 s version that only matched those same 15 s, while a 30 s\n    version that explains 18 s beats the 15 s cut-down inside it.\n    """\n    def score(a: Airing) -> float:\n        return a.aired_s + TIE_S * a.coverage\n\n    kept: list[Airing] = []\n    for a in sorted(airings, key=lambda a: (-score(a), a.name)):\n        host = next((k for k in kept\n                     if min(a.end_s, k.end_s) - max(a.start_s, k.start_s) > 0.5 * max(a.aired_s, 1e-6)),\n                    None)\n        if host is None:\n            kept.append(a)\n        elif a.name not in host.also_matched:\n            host.also_matched.append(a.name)\n    return kept\n', 'radioqc/naming.py': '"""Program name and start time from a broadcast file name.\n\nExample: \'CAO DIEM CHIEU 25.9.2026 16H30.mp3\' -> CAO DIEM CHIEU, 2026-09-25 16:30.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\nfrom datetime import date, datetime, time\nfrom pathlib import Path\n\n_DMY = re.compile(r"(?<!\\d)(\\d{1,2})[.\\-_/](\\d{1,2})[.\\-_/](\\d{4})(?!\\d)")\n_YMD = re.compile(r"(?<!\\d)(\\d{4})[.\\-_/](\\d{1,2})[.\\-_/](\\d{1,2})(?!\\d)")\n# After the date any separator is a time (\'16H30\', \'16:30\', \'16.30\'); before the date only an\n# explicit hour mark counts, because \'91.00\' / \'22.00\' there are usually part of the program name.\n_HM_AFTER = re.compile(r"(?<!\\d)(\\d{1,2})\\s*(?:[hHgG:.]|giờ)\\s*(\\d{2})(?!\\d)")\n_HM_BEFORE = re.compile(r"(?<!\\d)(\\d{1,2})\\s*(?:[hHgG]|giờ)\\s*(\\d{2})(?!\\d)")\n\n\n@dataclass\nclass BroadcastName:\n    program: str\n    day: date | None\n    start: datetime | None\n\n\ndef _find_day(stem: str) -> tuple[date | None, tuple[int, int] | None]:\n    for rx, day_first in ((_DMY, True), (_YMD, False)):\n        for m in rx.finditer(stem):\n            a, b, c = (int(g) for g in m.groups())\n            d, mo, y = (a, b, c) if day_first else (c, b, a)\n            try:\n                return date(y, mo, d), m.span()\n            except ValueError:\n                continue\n    return None, None\n\n\ndef parse_broadcast_name(filename: str) -> BroadcastName:\n    stem = Path(filename).stem\n    day, span = _find_day(stem)\n    if day is None or span is None:\n        return BroadcastName(stem.strip(), None, None)\n    parts = [stem[:span[0]], stem[span[1]:]]\n    start = None\n    # the first valid time after the date, else the valid time closest before it\n    for i, rx, order in ((1, _HM_AFTER, 1), (0, _HM_BEFORE, -1)):\n        for m in list(rx.finditer(parts[i]))[::order]:\n            hh, mm = int(m.group(1)), int(m.group(2))\n            if hh < 24 and mm < 60:\n                start = datetime.combine(day, time(hh, mm))\n                parts[i] = parts[i][:m.start()] + " " + parts[i][m.end():]\n                break\n        if start:\n            break\n    program = re.sub(r"[\\s_]+", " ", " ".join(parts)).strip(" -._")\n    return BroadcastName(program or stem.strip(), day, start)\n', 'radioqc/pipeline.py': '"""Batch processing of broadcast files on the Google Drive layout (mode A)."""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nimport traceback\nfrom dataclasses import dataclass\nfrom datetime import datetime\nfrom pathlib import Path\nfrom collections.abc import Callable\n\nimport numpy as np\n\nfrom . import __version__\nfrom .audio import AUDIO_EXTENSIONS, is_audio_file, load_audio\nfrom .fingerprint import FP_VERSION, SR, fingerprint\nfrom .library import LibraryIndex, build_library\nfrom .matcher import find_airings\nfrom .naming import parse_broadcast_name\nfrom .report import clock, details_rows, hms, summary_rows, write_report\n\nLog = Callable[[str], None]\n\n\n@dataclass\nclass Layout:\n    """Folders under the Drive root, e.g. /content/drive/MyDrive/RadioQC.\n\n    `uploads` is an optional folder outside Drive (Colab session storage) holding broadcast\n    files uploaded from the user\'s computer: processed like the inbox, never kept in Drive.\n    """\n    root: Path\n    uploads: Path | None = None\n\n    @property\n    def library(self) -> Path:\n        return self.root / "1_ThuVienQC"\n\n    @property\n    def inbox(self) -> Path:\n        return self.root / "2_FilePhatSong"\n\n    @property\n    def results(self) -> Path:\n        return self.root / "3_KetQua"\n\n    @property\n    def cache(self) -> Path:\n        return self.root / ".cache"\n\n    def ensure(self) -> None:\n        for p in (self.library, self.inbox, self.results, self.cache, self.uploads):\n            if p is not None:\n                p.mkdir(parents=True, exist_ok=True)\n\n\n@dataclass\nclass FileResult:\n    file: str\n    status: str\n    airings: int = 0\n    report: str = ""\n    seconds: float = 0.0\n    source: str = ""          # full path of the processed audio file\n\n\ndef broadcast_files(layout: Layout) -> list[Path]:\n    """Audio files from the Drive inbox and the upload folder; an uploaded file wins a name clash."""\n    found: dict[str, Path] = {}\n    for folder in (layout.inbox, layout.uploads):  # uploads last, so they replace same-named inbox files\n        if folder is not None and folder.is_dir():\n            found.update((p.name, p) for p in folder.iterdir() if is_audio_file(p))\n    return sorted(found.values(), key=lambda p: p.name.lower())\n\n\ndef save_uploads(uploaded: dict[str, bytes], folder: Path) -> tuple[list[Path], list[str]]:\n    """Write files from Colab\'s upload widget into `folder`; returns (saved audio files, skipped names)."""\n    folder.mkdir(parents=True, exist_ok=True)\n    saved, skipped = [], []\n    for raw_name, data in uploaded.items():\n        name = Path(raw_name.replace("\\\\", "/")).name  # never write outside `folder`\n        if not name or name.startswith(".") or Path(name).suffix.lower() not in AUDIO_EXTENSIONS:\n            skipped.append(raw_name)\n            continue\n        (folder / name).write_bytes(data)\n        saved.append(folder / name)\n    return saved, skipped\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=1), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef load_registry(layout: Layout) -> dict:\n    try:\n        return json.loads((layout.cache / "processed.json").read_text(encoding="utf-8"))\n    except (FileNotFoundError, json.JSONDecodeError):\n        return {}\n\n\ndef _stamp(p: Path) -> dict:\n    st = p.stat()\n    return {"size": st.st_size, "mtime_ns": st.st_mtime_ns}\n\n\ndef is_done(layout: Layout, registry: dict, p: Path) -> bool:\n    """Processed before, unchanged since, and its report still exists."""\n    r = registry.get(p.name)\n    if not r or {k: r.get(k) for k in ("size", "mtime_ns")} != _stamp(p):\n        return False\n    report = r.get("report", "")\n    return bool(report) and (layout.results / report).is_file()\n\n\ndef _has_spot_files(layout: Layout) -> bool:\n    return layout.library.is_dir() and any(is_audio_file(p) for p in layout.library.rglob("*"))\n\n\ndef self_check(layout: Layout) -> list[tuple[str, str, str]]:\n    """Rows of (item, \'OK\' | \'CHỜ FILE\' | \'CẢNH BÁO\' | \'LỖI\', detail) shown before running.\n\n    Empty folders are a normal state (each user adds their own files, if they have any),\n    so they read \'CHỜ FILE\'; \'LỖI\' is kept for real problems such as unreadable files.\n    """\n    rows = [("Thư mục gốc", "OK" if layout.root.is_dir() else "LỖI", str(layout.root))]\n    if _has_spot_files(layout):  # decode every spot now (cached), so a useless library shows up before running\n        index = build_library(layout.library, layout.cache / "library", log=lambda msg: None)\n        bad = "; ".join(f"{rel}: {why}" for rel, why in index.problems)\n        state = "LỖI" if not index.spots else ("CẢNH BÁO" if bad else "OK")\n        detail = f"{len(index.spots)} spot dùng được" + (f"; không dùng được: {bad}" if bad else "")\n    else:\n        state, detail = "CHỜ FILE", "chưa có file spot: bỏ file spot quảng cáo của bạn vào đây (tên file = tên spot)"\n    rows.append(("Thư viện QC (1_ThuVienQC)", state, detail))\n    files = broadcast_files(layout)\n    if not files:\n        rows.append(("File phát sóng (2_FilePhatSong)", "CHỜ FILE",\n                     "chưa có file: tải lên ở ô 2️⃣, hoặc bỏ file vào thư mục này"))\n        return rows\n    registry = load_registry(layout)\n    todo = sum(not is_done(layout, registry, p) for p in files)\n    rows.append(("File phát sóng (2_FilePhatSong)", "OK", f"{len(files)} file, {todo} file chưa xử lý"))\n    no_time = [p.name for p in files if parse_broadcast_name(p.name).start is None]\n    rows.append(("Tên file có ngày giờ", "CẢNH BÁO" if no_time else "OK",\n                 "không đọc được giờ bắt đầu (báo cáo sẽ dùng giờ tương đối): " + ", ".join(no_time)\n                 if no_time else "đọc được giờ bắt đầu của mọi file"))\n    return rows\n\n\ndef next_step(layout: Layout) -> str | None:\n    """What the user should do next while the spot library is empty; None once it has files.\n\n    Broadcast files normally arrive through the upload cell, so an empty inbox needs no nudge.\n    """\n    if _has_spot_files(layout):\n        return None\n    if not broadcast_files(layout):\n        return ("✅ Thư mục RadioQC đã sẵn sàng trong Google Drive của bạn. Bước tiếp theo: bỏ file spot quảng cáo "\n                "của bạn vào 1_ThuVienQC (chỉ cần làm một lần), rồi Run all và tải file phát sóng lên ở ô 2️⃣ "\n                "(hoặc bỏ sẵn vào 2_FilePhatSong).")\n    return "Bước tiếp theo: bỏ file spot quảng cáo của bạn vào 1_ThuVienQC (tên file = tên spot), rồi Run all lần nữa."\n\n\ndef run_mode_a(layout: Layout, rerun: bool = False, log: Log = print) -> list[FileResult]:\n    """Process every new broadcast file; one failing file never stops the batch."""\n    layout.ensure()\n    t0 = time.time()\n    index = build_library(layout.library, layout.cache / "library", log=log)\n    for rel, why in index.problems:\n        log(f"⚠️ Thư viện: {rel}: {why}")\n    if not index.spots:\n        if index.problems:\n            log("❌ Không có file spot nào dùng được trong 1_ThuVienQC (xem cảnh báo ở trên): dừng, không tạo báo cáo.")\n        else:\n            log("ℹ️ Chưa có file spot trong 1_ThuVienQC nên chưa có gì để dò. Bỏ file spot vào rồi chạy lại.")\n        return []\n    files = broadcast_files(layout)\n    if not files:\n        log("ℹ️ Chưa có file phát sóng: tải lên ở ô 2️⃣ hoặc bỏ vào 2_FilePhatSong, rồi chạy lại.")\n        return []\n    log(f"Thư viện: {len(index.spots)} spot ({time.time() - t0:.0f} s)")\n    registry = load_registry(layout)\n    results = []\n    for p in files:\n        if not rerun and is_done(layout, registry, p):\n            r = registry[p.name]\n            results.append(FileResult(p.name, "Bỏ qua (đã xử lý)", r.get("airings", 0), r["report"],\n                                      source=str(p)))\n            continue\n        t1 = time.time()\n        out = _out_stem(p, files)\n        try:\n            n = _process(layout, index, p, out, log)\n        except Exception as e:  # noqa: BLE001 - report it and continue with the next file\n            log(f"❌ {p.name}: {e}\\n{traceback.format_exc()}")\n            results.append(FileResult(p.name, f"LỖI: {e}", seconds=time.time() - t1, source=str(p)))\n            continue\n        registry[p.name] = {**_stamp(p), "report": out + ".xlsx", "airings_file": out + ".json", "path": str(p),\n                            "airings": n, "processed_at": datetime.now().isoformat(timespec="seconds")}\n        _write_json(layout.cache / "processed.json", registry)\n        results.append(FileResult(p.name, "OK", n, out + ".xlsx", time.time() - t1, source=str(p)))\n    return results\n\n\ndef _out_stem(p: Path, files: list[Path]) -> str:\n    """Output name for `p`; \'X.mp3\' and \'X.wav\' get \'X (mp3)\' and \'X (wav)\' so they never collide."""\n    twins = sum(f.stem.lower() == p.stem.lower() for f in files)\n    return p.stem if twins < 2 else f"{p.stem} ({p.suffix.lstrip(\'.\').lower()})"\n\n\ndef _process(layout: Layout, index: LibraryIndex, p: Path, out: str, log: Log) -> int:\n    name = parse_broadcast_name(p.name)\n    x = load_audio(p, SR)\n    duration = x.size / SR\n    fp = fingerprint(x)\n    del x\n    airings = find_airings(index, fp)\n    info = [\n        ("File", p.name),\n        ("Chương trình", name.program),\n        ("Bắt đầu (đọc từ tên file)", name.start.strftime("%d/%m/%Y %H:%M") if name.start\n         else "KHÔNG ĐỌC ĐƯỢC: giờ trong báo cáo là giờ tương đối tính từ đầu file"),\n        ("Thời lượng file", hms(duration)),\n        ("Số spot trong thư viện", str(len(index.spots))),\n        ("Số lần phát tìm thấy", str(len(airings))),\n        ("Chế độ", "A – Đối soát"),\n        ("Phiên bản", f"radioqc {__version__}, vân tay v{FP_VERSION}"),\n        ("Xử lý lúc", datetime.now().strftime("%d/%m/%Y %H:%M:%S")),\n    ] + [("Cảnh báo thư viện", f"{rel}: {why}") for rel, why in index.problems]\n    write_report(layout.results / (out + ".xlsx"), details_rows(airings, index, name),\n                 summary_rows(airings, index, name), info)\n    _write_json(layout.cache / "airings" / (out + ".json"),\n                [{"stt": k, "name": a.name, "start_s": a.start_s, "end_s": a.end_s,\n                  "clock": clock(name, a.start_s)[1]} for k, a in enumerate(airings, 1)])\n    log(f"✅ {p.name}: dài {hms(duration)}, {len(airings)} lần phát → 3_KetQua/{out}.xlsx")\n    return len(airings)\n\n\ndef listen_clip(layout: Layout, file: str = "", stt: int = 1, pad_s: float = 1.0,\n                sr: int = 16000) -> tuple[np.ndarray, int, str]:\n    """Audio of airing `stt` in `file` (default: the most recently processed file)."""\n    registry = load_registry(layout)\n    if not registry:\n        raise ValueError("Chưa có lần phát nào để nghe: tải file phát sóng lên rồi chạy ô 3️⃣ trước.")\n    if not file:\n        file = max(registry, key=lambda k: registry[k].get("processed_at", ""))\n    file = file.strip()\n    if file not in registry:  # accept the name without its extension too\n        same = [k for k in registry if Path(k).stem == file]\n        if len(same) > 1:\n            raise ValueError(f"Có nhiều file tên \'{file}\': {\', \'.join(sorted(same))}. Hãy gõ cả phần đuôi.")\n        file = same[0] if same else file\n    if file not in registry:\n        raise ValueError(f"Không thấy \'{file}\'. Các file đã xử lý: {\', \'.join(sorted(registry))}")\n    sidecar = registry[file].get("airings_file") or Path(file).stem + ".json"\n    rows = json.loads((layout.cache / "airings" / sidecar).read_text(encoding="utf-8"))\n    if not 1 <= stt <= len(rows):\n        raise ValueError(f"\'{file}\' có {len(rows)} lần phát: STT phải từ 1 đến {len(rows)}.")\n    audio = Path(registry[file].get("path") or layout.inbox / file)\n    if not audio.is_file():\n        raise ValueError(f"File \'{file}\' không còn nên không nghe lại được (file tải lên chỉ tồn tại trong phiên "\n                         "Colab lúc tải; hãy tải lên lại, hoặc để file trong 2_FilePhatSong).")\n    r = rows[stt - 1]\n    start = max(0.0, r["start_s"] - pad_s)\n    x = load_audio(audio, sr, start_s=start, duration_s=r["end_s"] + pad_s - start)\n    return x, sr, f"{file}: STT {stt} — {r[\'name\']} lúc {r[\'clock\']}"\n', 'radioqc/repeats.py': '"""Stretches of audio that repeat inside one recording: ads usually air more than once.\n\nThe recording\'s landmark hashes are matched against themselves; a repeat shows up as many\npairs agreeing on one time offset. Proven on the 25/9 broadcast (spec §2).\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .fingerprint import DT_MAX, FPS, Fingerprint\n\nMIN_GAP_S = 30.0   # ignore self-similarity closer than this (musical structure inside one item)\nMAX_OCC = 20       # hashes more common than this carry no information (hum, tones)\nMIN_PAIRS = 12     # landmark pairs agreeing on one offset, summed over +/-1 frame\nRUN_PAIRS = 10     # pairs needed in one continuous run\nRUN_GAP_S = 4.0    # a run may pause this long (quiet passage inside an ad) and still be one stretch\nMIN_LEN_S = 3.0\nJOIN_OVERLAP_S = 1.0  # consecutive pieces of one ad may overlap this much, or be RUN_GAP_S apart\n\n\n@dataclass\nclass Repeat:\n    occurrences: list[tuple[float, float]]   # (start_s, end_s), sorted by start\n\n    @property\n    def count(self) -> int:\n        return len(self.occurrences)\n\n    @property\n    def duration_s(self) -> float:\n        return float(np.median([b - a for a, b in self.occurrences]))\n\n\ndef _self_pairs(fp: Fingerprint) -> tuple[np.ndarray, np.ndarray]:\n    """(offset, anchor time) in frames for every pair of equal hashes at least MIN_GAP_S apart."""\n    o = np.lexsort((fp.times, fp.hashes))\n    h, t = fp.hashes[o], fp.times[o].astype(np.int64)\n    starts = np.flatnonzero(np.r_[True, h[1:] != h[:-1]])\n    sizes = np.diff(np.r_[starts, h.size])\n    usable = np.repeat(sizes, sizes) <= MAX_OCC\n    offsets, anchors = [], []\n    for k in range(1, MAX_OCC):\n        i = np.arange(h.size - k)\n        i = i[(h[i + k] == h[i]) & usable[i]]\n        d = t[i + k] - t[i]\n        far = d >= MIN_GAP_S * FPS\n        offsets.append(d[far])\n        anchors.append(t[i][far])\n    return np.concatenate(offsets), np.concatenate(anchors)\n\n\ndef _segment_pairs(offsets: np.ndarray, anchors: np.ndarray) -> list[tuple[float, float, float]]:\n    """(start_s, end_s, offset_s): a stretch and its copy `offset_s` later."""\n    if offsets.size == 0:\n        return []\n    cnt = np.bincount(offsets)\n    c3 = cnt.copy()\n    c3[1:] += cnt[:-1]\n    c3[:-1] += cnt[1:]\n    padded = np.r_[np.zeros(2, c3.dtype), c3, np.zeros(2, c3.dtype)]\n    local_max = np.max(np.lib.stride_tricks.sliding_window_view(padded, 5), axis=1) == c3\n    order = np.argsort(offsets, kind="stable")\n    so, sa = offsets[order], anchors[order]\n    out = []\n    for d in np.flatnonzero((c3 >= MIN_PAIRS) & local_max):\n        lo, hi = np.searchsorted(so, [d - 1, d + 2])\n        a = np.sort(sa[lo:hi])\n        for run in np.split(a, np.flatnonzero(np.diff(a) > RUN_GAP_S * FPS) + 1):\n            s0, s1 = run[0] / FPS, (run[-1] + DT_MAX) / FPS\n            if run.size >= RUN_PAIRS and s1 - s0 >= MIN_LEN_S:\n                out.append((s0, s1, d / FPS))\n    return out\n\n\ndef _find(parent: list[int], a: int) -> int:\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _join_pieces(repeats: list[Repeat]) -> list[Repeat]:\n    """One ad found as consecutive pieces, because its airings drift apart mid-way (a slightly different cut\n    moves the offset by ~1 s, seen on 25/9): same number of airings, and in every airing the second piece\n    starts where the first ends."""\n    out = sorted(repeats, key=lambda r: r.occurrences[0][0])\n    joined = True\n    while joined:\n        joined = False\n        for r in out:\n            nxt = next((q for q in out if q is not r and q.count == r.count and all(\n                -JOIN_OVERLAP_S <= q0 - r1 <= RUN_GAP_S for (_, r1), (q0, _) in zip(r.occurrences, q.occurrences))),\n                None)\n            if nxt is not None:\n                r.occurrences = [(r0, max(r1, q1)) for (r0, r1), (_, q1) in zip(r.occurrences, nxt.occurrences)]\n                out.remove(nxt)\n                joined = True\n                break\n    return out\n\n\ndef find_repeats(fp: Fingerprint) -> list[Repeat]:\n    """Every stretch heard at least twice in the recording, grouped by content."""\n    pairs = _segment_pairs(*_self_pairs(fp))\n    if not pairs:\n        return []\n    iv = np.array([v for s0, s1, d in pairs for v in ((s0, s1), (s0 + d, s1 + d))])\n    parent = list(range(len(iv)))\n    for p in range(0, len(iv), 2):                 # the two copies of a pair are the same content\n        parent[_find(parent, p)] = _find(parent, p + 1)\n    order = np.argsort(iv[:, 0])\n    for x, a in enumerate(order):                  # overlapping stretches are the same occurrence\n        for b in order[x + 1:]:\n            if iv[b, 0] >= iv[a, 1]:\n                break\n            overlap = min(iv[a, 1], iv[b, 1]) - max(iv[a, 0], iv[b, 0])\n            if overlap > 0.5 * min(iv[a, 1] - iv[a, 0], iv[b, 1] - iv[b, 0]):\n                parent[_find(parent, a)] = _find(parent, b)\n    groups: dict[int, list[tuple[float, float]]] = {}\n    for k, (s0, s1) in enumerate(iv):\n        groups.setdefault(_find(parent, k), []).append((float(s0), float(s1)))\n    repeats = []\n    for spans in groups.values():\n        merged: list[list[float]] = []\n        for s0, s1 in sorted(spans):\n            if merged and s0 <= merged[-1][1] - 0.5 * (s1 - s0):\n                merged[-1][1] = max(merged[-1][1], s1)\n            else:\n                merged.append([s0, s1])\n        if len(merged) >= 2:\n            repeats.append(Repeat([(round(a, 2), round(b, 2)) for a, b in merged]))\n    return _join_pieces(repeats)\n', 'radioqc/report.py': '"""Excel report for one broadcast file (openpyxl only: pure Python, no compiled deps)."""\nfrom __future__ import annotations\n\nimport os\nfrom datetime import timedelta\nfrom pathlib import Path\n\nfrom openpyxl import Workbook\nfrom openpyxl.styles import Alignment, Font\n\nfrom .library import LibraryIndex\nfrom .matcher import Airing\nfrom .naming import BroadcastName\n\nDETAIL_COLUMNS = ["STT", "Ngày", "Giờ bắt đầu", "Giờ kết thúc", "Thời lượng phát (s)", "Tên spot",\n                  "Thời lượng spot (s)", "Mức khớp (%)", "Tình trạng", "Cũng khớp", "Vị trí trong file"]\nSUMMARY_COLUMNS = ["Tên spot", "Số lần phát", "Các giờ phát", "Số lần phát thiếu", "Số lần cần nghe lại"]\n\n\ndef hms(seconds: float) -> str:\n    s = int(round(seconds))\n    return f"{s // 3600:02d}:{s % 3600 // 60:02d}:{s % 60:02d}"\n\n\ndef clock(name: BroadcastName, seconds: float) -> tuple[str, str]:\n    """(date, time) of a position in the file; \'+hh:mm:ss\' when the start time is unknown."""\n    if name.start is None:\n        return (name.day.strftime("%d/%m/%Y") if name.day else ""), "+" + hms(seconds)\n    t = name.start + timedelta(seconds=int(round(seconds)))\n    return t.strftime("%d/%m/%Y"), t.strftime("%H:%M:%S")\n\n\ndef details_rows(airings: list[Airing], index: LibraryIndex, name: BroadcastName) -> list[dict]:\n    rows = []\n    for k, a in enumerate(airings, 1):\n        day, start = clock(name, a.start_s)\n        _, end = clock(name, a.end_s)\n        rows.append(dict(zip(DETAIL_COLUMNS, [\n            k, day, start, end, round(a.aired_s, 1), a.name,\n            round(index.spots[a.spot_id].duration_s, 1), int(round(100 * a.coverage)),\n            a.status, ", ".join(a.also_matched), hms(a.start_s)], strict=True)))\n    return rows\n\n\ndef summary_rows(airings: list[Airing], index: LibraryIndex, name: BroadcastName) -> list[dict]:\n    """One row per library spot (0 airings included), most aired first."""\n    rows = []\n    for sid, spot in enumerate(index.spots):\n        mine = [a for a in airings if a.spot_id == sid]\n        rows.append(dict(zip(SUMMARY_COLUMNS, [\n            spot.name, len(mine), ", ".join(clock(name, a.start_s)[1] for a in mine),\n            sum(a.status.startswith("Phát thiếu") for a in mine),\n            sum(a.status.startswith("Cần nghe lại") for a in mine)], strict=True)))\n    return sorted(rows, key=lambda r: (-r["Số lần phát"], r["Tên spot"]))\n\n\ndef _sheet(wb: Workbook, title: str, columns: list[str], rows: list[dict]) -> None:\n    ws = wb.create_sheet(title)\n    ws.append(columns)\n    for r in rows:\n        ws.append([r[c] for c in columns])\n        for cell in ws[ws.max_row]:\n            if isinstance(cell.value, str) and cell.value.startswith("="):\n                cell.data_type = "s"  # a file named \'=...\' stays text, never a formula\n    for cell in ws[1]:\n        cell.font = Font(bold=True)\n    ws.freeze_panes = "A2"\n    for col in ws.columns:\n        width = max(len(str(c.value)) for c in col if c.value is not None)\n        ws.column_dimensions[col[0].column_letter].width = min(60, width + 2)\n        if width > 60:  # long text (what was said) wraps instead of running off the screen\n            for c in col[1:]:\n                c.alignment = Alignment(wrap_text=True, vertical="top")\n\n\ndef write_workbook(path: Path, sheets: list[tuple[str, list[str], list[dict]]]) -> None:\n    """Write (title, columns, rows) sheets atomically, so a half-written report never appears."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    wb = Workbook()\n    wb.remove(wb.active)\n    for title, columns, rows in sheets:\n        _sheet(wb, title, columns, rows)\n    tmp = path.with_name(path.stem + ".tmp.xlsx")\n    wb.save(tmp)\n    os.replace(tmp, path)\n\n\ndef info_rows(info: list[tuple[str, str]]) -> list[dict]:\n    return [{"Mục": k, "Giá trị": v} for k, v in info]\n\n\ndef write_report(path: Path, details: list[dict], summary: list[dict],\n                 info: list[tuple[str, str]]) -> None:\n    """Mode A report on the Drive layout: Chi tiết, Tổng hợp, Thông tin."""\n    write_workbook(path, [("Chi tiết", DETAIL_COLUMNS, details), ("Tổng hợp", SUMMARY_COLUMNS, summary),\n                          ("Thông tin", ["Mục", "Giá trị"], info_rows(info))])\n\n\n# --- one-button report (spots found by fingerprint and/or by AI): one workbook per broadcast ---\n\nAD_COLUMNS = ["STT", "Ngày", "Giờ bắt đầu", "Giờ kết thúc", "Thời lượng (s)", "Nhãn hàng / spot", "Sản phẩm",\n              "Loại", "Lời quảng cáo", "Căn cứ", "Cần nghe lại", "Vị trí trong file"]\nBRAND_COLUMNS = ["Nhãn hàng / spot", "Số lần phát", "Các giờ phát", "Tổng thời lượng (s)"]\nTRANSCRIPT_COLUMNS = ["Giờ", "Vị trí trong file", "Lời", "Thuộc quảng cáo"]\nNOT_ANALYSED = "⚠️ CHƯA PHÂN TÍCH"\nLOST = "⚠️ KHÔNG CHÉP ĐƯỢC LỜI"\nUNKNOWN_BRAND = "(chưa rõ nhãn hàng)"\n\n\ndef ad_rows(detections, name: BroadcastName, not_analysed: list[tuple[float, float]] = ()) -> list[dict]:\n    """Every ad, plus a loud row for each stretch the AI could not analyse (never a silent gap)."""\n    items = [(d.start_s, d.end_s, d) for d in detections] + [(a, b, None) for a, b in not_analysed]\n    rows = []\n    for k, (start, end, d) in enumerate(sorted(items, key=lambda t: t[0]), 1):\n        day, t0 = clock(name, start)\n        _, t1 = clock(name, end)\n        if d is None:\n            values = [NOT_ANALYSED, "", "", "", "",\n                      "AI không trả lời được đoạn này: quảng cáo trong đoạn có thể bị bỏ sót, hãy chạy lại"]\n        else:\n            values = [d.brand or UNKNOWN_BRAND, d.product, d.category, d.text, "; ".join(d.evidence), d.review]\n        rows.append(dict(zip(AD_COLUMNS, [k, day, t0, t1, round(end - start, 1), *values, hms(start)], strict=True)))\n    return rows\n\n\ndef brand_rows(detections, name: BroadcastName) -> list[dict]:\n    groups: dict[str, list] = {}\n    for d in detections:\n        groups.setdefault((d.brand or UNKNOWN_BRAND).strip().lower(), []).append(d)\n    rows = [dict(zip(BRAND_COLUMNS, [ds[0].brand or UNKNOWN_BRAND, len(ds),\n                                     ", ".join(clock(name, d.start_s)[1] for d in ds),\n                                     round(sum(d.duration_s for d in ds), 1)], strict=True))\n            for ds in groups.values()]\n    return sorted(rows, key=lambda r: (-r["Số lần phát"], r["Nhãn hàng / spot"].lower()))\n\n\ndef transcript_rows(lines, detections, name: BroadcastName, lost: list[tuple[float, float]] = ()) -> list[dict]:\n    """Every line in time order, with a loud row where speech could not be transcribed."""\n    items = [(ln.start, ln.end, ln.text) for ln in lines] + [\n        (a, b, f"{LOST} ({b - a:.0f} giây): thường là bài hát; nếu cần chắc chắn thì nghe lại đoạn này")\n        for a, b in lost]\n    rows = []\n    for start, end, text in sorted(items, key=lambda t: t[0]):\n        mid = (start + end) / 2\n        ad = next((d for d in detections if d.start_s <= mid <= d.end_s), None)\n        rows.append(dict(zip(TRANSCRIPT_COLUMNS, [clock(name, start)[1], hms(start), text,\n                                                  (ad.brand or UNKNOWN_BRAND) if ad else ""], strict=True)))\n    return rows\n', 'radioqc/simple.py': '"""The one-button flow for non-technical users: every picked file in, one Excel per broadcast out.\n\nFiles up to SPOT_MAX_S long are the user\'s own spots (checked exactly by fingerprint, mode A);\nlonger files are broadcasts. With a GPU the broadcasts are also transcribed and read by the AI\n(mode B), which finds ads nobody supplied.\n"""\nfrom __future__ import annotations\n\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport unicodedata\nfrom collections.abc import Callable\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom . import __version__\nfrom .audio import duration_of, is_audio_file, load_audio\nfrom .classify import LLM, classify\nfrom .clips import audio_tag, mp3_clip\nfrom .discover import Detection, detect\nfrom .fingerprint import FP_VERSION, FPS, SR, Fingerprint, fingerprint\nfrom .learn import PENDING_REVIEW, PENDING_SUBDIR, candidates, save_spots, verified\nfrom .library import MIN_SPOT_S, LibraryIndex\nfrom .matcher import find_airings\nfrom .naming import BroadcastName, parse_broadcast_name\nfrom .repeats import find_repeats\nfrom .report import (AD_COLUMNS, BRAND_COLUMNS, TRANSCRIPT_COLUMNS, UNKNOWN_BRAND, ad_rows, brand_rows, clock,\n                     hms, info_rows, transcript_rows, write_workbook)\nfrom .transcript import ASR_MODEL, ASR_SR, Line\nfrom .ui import html_note, html_table\n\nSPOT_MAX_S = 300.0\nUPLOAD_DIR = Path("/content/tai_len")\nRESULT_DIR = Path("/content/ket_qua")\nDRIVE_ROOT = Path("/content/drive/MyDrive")\nSPOT_SUBDIR, BROADCAST_SUBDIR, RESULT_SUBDIR = "1_ThuVienQC", "2_FilePhatSong", "3_KetQua"\n# Free Drive is 15 GB shared with Gmail and Photos, ~2-3 weeks of 5 three-hour files a day: past this much,\n# processed broadcasts get a hint to delete them (their reports stay). Nothing is ever deleted for the user.\nDRIVE_NOTE_BYTES = 2 * 1024**3\nSpan = tuple[float, float]\n# audio, progress -> (lines, stretches that could not be transcribed)\nASR = Callable[[np.ndarray, Callable[[float], None]], tuple[list[Line], list[Span]]]\nLog = Callable[[str], None]\n\n\n@dataclass\nclass Outcome:\n    file: str\n    status: str                     # "OK", "LỖI: …" or "BỎ QUA: …"\n    name: BroadcastName | None = None\n    report: Path | None = None\n    detections: list[Detection] = field(default_factory=list)\n    not_analysed: list[Span] = field(default_factory=list)\n    lost: list[Span] = field(default_factory=list)       # speech Whisper could not transcribe\n    asr_error: str = ""                                   # the transcriber failed: only spot results\n    clips: list[str] = field(default_factory=list)\n    seconds: float = 0.0\n    learned: list[str] = field(default_factory=list)     # new spot files, waiting in 1_ThuVienQC/Chờ duyệt\n\n\n@dataclass\nclass DriveInbox:\n    """What to do with the user\'s Drive folder this run."""\n    spot_dir: Path\n    result_dir: Path\n    broadcasts: list[Path]     # to process now\n    done: list[Path]           # skipped: their report is already in result_dir\n    spots: int                 # audio files in spot_dir\n    message: str\n\n\ndef report_path(out_dir: Path, broadcast: Path) -> Path:\n    return out_dir / f"{broadcast.stem} - KetQua.xlsx"\n\n\ndef _audio_in(folder: Path) -> list[Path]:\n    return sorted((p for p in folder.rglob("*") if is_audio_file(p)), key=lambda p: p.name.lower())\n\n\ndef drive_inbox(drive_root: Path, folder: str, redo: bool = False) -> DriveInbox:\n    """Broadcasts in `folder`/2_FilePhatSong on the user\'s Drive that have no report in 3_KetQua yet (all of\n    them with `redo`). Creates the folders on first use: an empty inbox is a normal state, not an error."""\n    folder = folder.strip().strip("/\\\\").strip() or "RadioQC"\n    root = drive_root / folder\n    created = not (root / BROADCAST_SUBDIR).is_dir()\n    spot_dir, inbox, result_dir = root / SPOT_SUBDIR, root / BROADCAST_SUBDIR, root / RESULT_SUBDIR\n    for d in (spot_dir, inbox, result_dir):\n        d.mkdir(parents=True, exist_ok=True)\n    audio = _audio_in(inbox)\n    done = [] if redo else [p for p in audio if report_path(result_dir, p).is_file()]\n    todo = [p for p in audio if p not in done]\n    spots = len(_audio_in(spot_dir))\n    where = f"{folder}/{BROADCAST_SUBDIR}"\n    if created:\n        message = (f"Đã tạo thư mục {folder} trong Google Drive của bạn. Bỏ file phát sóng vào {where} (file spot "\n                   f"quảng cáo, nếu có, vào {folder}/{SPOT_SUBDIR}) rồi Run all lại.")\n    elif not audio:\n        message = f"CHỜ FILE: thư mục {where} trong Drive chưa có file âm thanh nào. Bỏ file vào đó rồi Run all lại."\n    elif not todo:\n        message = (f"Không có file mới: cả {len(done)} file trong {where} đã có kết quả trong {folder}/{RESULT_SUBDIR}."\n                   " Bỏ file mới vào rồi Run all lại, hoặc tick CHAY_LAI_TAT_CA ở ô ⚙️ Tuỳ chọn để chạy lại.")\n    else:\n        message = (f"Sẽ xử lý {len(todo)} file: {\', \'.join(p.name for p in todo)}"\n                   + (f" (bỏ qua {len(done)} file đã có kết quả)" if done else "")\n                   + (f". Dùng {spots} file spot trong {folder}/{SPOT_SUBDIR}" if spots else "") + ".")\n    kept = sum(p.stat().st_size for p in done)\n    if kept >= DRIVE_NOTE_BYTES:\n        message += (f" ℹ️ {len(done)} file đã xử lý đang chiếm {kept / 1024**3:.1f} GB Drive: xoá bớt nếu Drive sắp đầy"\n                    f" (kết quả vẫn còn trong {folder}/{RESULT_SUBDIR}).")\n    return DriveInbox(spot_dir, result_dir, todo, done, spots, message)\n\n\ndef split_by_length(files: list[Path]) -> tuple[list[Path], list[Path]]:\n    """(broadcasts, spots): anything up to SPOT_MAX_S is a spot. Unreadable files count as broadcasts\n    so that processing reports them as errors instead of dropping them."""\n    broadcasts, spots = [], []\n    for p in files:\n        try:\n            short = duration_of(p) <= SPOT_MAX_S\n        except Exception:  # noqa: BLE001 - reported later, when the file is processed\n            short = False\n        (spots if short else broadcasts).append(p)\n    return broadcasts, spots\n\n\ndef spot_library(spots: list[Path], extra: LibraryIndex | None = None) -> LibraryIndex:\n    """Index the picked spot files, plus an optional library already built (e.g. from Drive)."""\n    items, problems = [], list(extra.problems) if extra else []\n    if extra:\n        items += [(s.name, s.file, Fingerprint(s.hashes, s.times, int(round(s.duration_s * FPS)), s.duration_s))\n                  for s in extra.spots]\n    for p in spots:\n        try:\n            fp = fingerprint(load_audio(p, SR))\n        except Exception as e:  # noqa: BLE001 - list every unreadable file, keep going\n            problems.append((p.name, f"không đọc được file ({e})"))\n            continue\n        if fp.duration_s < MIN_SPOT_S:\n            problems.append((p.name, f"quá ngắn ({fp.duration_s:.1f} s < {MIN_SPOT_S:.0f} s) để dò tin cậy"))\n        elif fp.hashes.size == 0:\n            problems.append((p.name, "không có nội dung để lấy vân tay (im lặng?)"))\n        else:\n            items.append((p.stem, p.name, fp))\n    return LibraryIndex.from_fingerprints(items, problems)\n\n\ndef _progress(log: Log, label: str) -> Callable[[float], None]:\n    shown = [-1]\n\n    def report(fraction: float) -> None:\n        step = int(fraction * 10)\n        if step > shown[0]:\n            shown[0] = step\n            log(f"   {label}: {10 * step}%")\n    return report\n\n\ndef process(path: Path, index: LibraryIndex, asr: ASR | None, llm: LLM | None, out_dir: Path,\n            log: Log = print, learn_dir: Path | None = None) -> Outcome:\n    t0 = time.time()\n    name = parse_broadcast_name(path.name)\n    x8 = load_audio(path, SR)\n    duration = x8.size / SR\n    fp = fingerprint(x8)\n    airings = find_airings(index, fp) if index.spots else []\n    lines: list[Line] = []\n    spans, repeats, not_analysed, lost = [], [], [], []\n    asr_error = ""\n    if asr is not None:\n        log("   Đang chép lời (bước lâu nhất)…")\n        try:\n            lines, lost = asr(load_audio(path, ASR_SR), _progress(log, "Chép lời"))\n        except Exception as e:  # noqa: BLE001 - the spot results must survive a broken transcriber\n            asr_error = f"{type(e).__name__}: {e}"\n            log(f"⚠️ Không chép lời được: {asr_error}")\n            not_analysed = [(0.0, duration)]\n        repeats = find_repeats(fp)\n        in_repeat = {i for i, ln in enumerate(lines)\n                     if any(a < ln.end and ln.start < b for r in repeats for a, b in r.occurrences)}\n        if llm is not None and lines:\n            log("   AI đang đọc bản chép lời…")\n            result = classify(lines, llm, in_repeat, log)\n            spans, not_analysed = result.spans, result.failed\n        elif lines:\n            not_analysed = [(0.0, duration)]\n    detections = detect(lines, spans, repeats, airings)\n    pending = {s.name for s in index.spots\n               if unicodedata.normalize("NFC", s.file).startswith(f"{PENDING_SUBDIR}/")}\n    for d in detections:\n        if d.source == "A" and d.brand in pending:\n            d.review = f"{d.review}; {PENDING_REVIEW}" if d.review else PENDING_REVIEW\n    learned = []\n    if learn_dir is not None:\n        keep = verified(x8, fp, candidates(detections, repeats))\n        learned = [p.name for p in save_spots(x8, SR, keep, name, learn_dir)]\n        if learned:\n            log(f"   Lưu {len(learned)} quảng cáo mới làm spot (chờ bạn duyệt)")\n    how = []\n    if index.spots:\n        how.append(f"so vân tay với {len(index.spots)} spot của bạn (chính xác)")\n    if asr is not None:\n        how.append("AI tự nhận diện từ lời nói (nên nghe lại để xác nhận)")\n    info = [\n        ("File", path.name),\n        ("Chương trình", name.program),\n        ("Bắt đầu (đọc từ tên file)", name.start.strftime("%d/%m/%Y %H:%M") if name.start\n         else "KHÔNG ĐỌC ĐƯỢC: giờ trong báo cáo là giờ tính từ đầu file"),\n        ("Thời lượng file", hms(duration)),\n        ("Cách dò", "; ".join(how)),\n        ("Số quảng cáo tìm thấy", str(len(detections))),\n        ("Đoạn chưa phân tích", ", ".join(f"{hms(a)}–{hms(b)}" for a, b in not_analysed) or "không có"),\n        ("Chép lời", f"LỖI: {asr_error}" if asr_error else f"Whisper {ASR_MODEL}" if asr is not None\n         else "không (máy không có GPU)"),\n        ("Đoạn không chép được lời", f"{len(lost)} đoạn, tổng {sum(b - a for a, b in lost):.0f} giây: "\n         + ", ".join(f"{hms(a)}–{hms(b)}" for a, b in lost) if lost else "không có"),\n        ("Spot tự học mới", (f"{len(learned)} file trong 1_ThuVienQC/{PENDING_SUBDIR}: " + ", ".join(learned))\n         if learned else "không có" if learn_dir is not None else "không học (không dùng Drive)"),\n        ("Thời gian xử lý", f"{time.time() - t0:.0f} giây"),\n        ("Phiên bản", f"radioqc {__version__}, vân tay v{FP_VERSION}"),\n    ] + [("Cảnh báo spot", f"{rel}: {why}") for rel, why in index.problems]\n    if lines or lost:\n        transcript = transcript_rows(lines, detections, name, lost)\n    else:\n        why = ("Không chép lời vì máy Colab lần này không có GPU." if asr is None\n               else f"Không chép lời được: {asr_error}" if asr_error else "Không nghe thấy lời nói nào.")\n        transcript = [dict.fromkeys(TRANSCRIPT_COLUMNS, "") | {"Lời": why}]\n    report = report_path(out_dir, path)\n    write_workbook(report, [("Quảng cáo", AD_COLUMNS, ad_rows(detections, name, not_analysed)),\n                            ("Tổng hợp", BRAND_COLUMNS, brand_rows(detections, name)),\n                            ("Chép lời", TRANSCRIPT_COLUMNS, transcript),\n                            ("Thông tin", ["Mục", "Giá trị"], info_rows(info))])\n    clips = [audio_tag(mp3_clip(x8, SR, d.start_s, d.end_s)) for d in detections]\n    return Outcome(path.name, "OK", name, report, detections, not_analysed, lost, asr_error, clips,\n                   time.time() - t0, learned)\n\n\ndef run(files: list[Path], out_dir: Path, *, extra_library: LibraryIndex | None = None, asr: ASR | None = None,\n        llm: LLM | None = None, log: Log = print, learn_dir: Path | None = None) -> list[Outcome]:\n    broadcasts, spots = split_by_length(files)\n    index = spot_library(spots, extra_library)\n    for rel, why in index.problems:\n        log(f"⚠️ Spot {rel}: {why}")\n    if spots or extra_library:\n        log(f"Spot của bạn: {len(index.spots)} file dùng được.")\n    if not broadcasts:\n        log("ℹ️ Chưa có file phát sóng (file dài hơn 5 phút). Run all lần nữa và chọn file phát sóng.")\n        return []\n    if not index.spots and asr is None:\n        why = ("máy Colab lần này không có GPU để AI tự nhận diện, và không có file spot để so. Cách sửa: bấm "\n               "Runtime → Change runtime type → chọn T4 GPU → Save, rồi Run all lại (hoặc chọn kèm file spot "\n               "quảng cáo để dò mà không cần GPU)")\n        log(f"❌ Chưa dò được: {why}.")\n        return [Outcome(p.name, f"BỎ QUA: {why}") for p in broadcasts]\n    if asr is not None and llm is None:\n        log("⚠️ Không dùng được Gemini của Colab (google.colab.ai) nên AI chưa tự tìm quảng cáo: file Excel vẫn có"\n            " bản chép lời, và kết quả spot nếu bạn có chọn file spot.")\n    outcomes = []\n    for p in broadcasts:\n        log(f"▶ {p.name}")\n        try:\n            outcomes.append(process(p, index, asr, llm, out_dir, log, learn_dir))\n        except Exception as e:  # noqa: BLE001 - one bad file must not stop the others\n            log(f"❌ {p.name}: {e}\\n{traceback.format_exc()}")\n            outcomes.append(Outcome(p.name, f"LỖI: {e}"))\n    return outcomes\n\n\ndef results_html(o: Outcome) -> str:\n    """The results table shown under the run cell: one row per ad, each with a ▶ player."""\n    if o.status != "OK":\n        return html_note(f"{o.file}: {o.status}")\n    head = f"<p><b>{o.file}</b>: {len(o.detections)} quảng cáo · file Excel: {o.report.name}</p>"\n    rows = [(k, clock(o.name, d.start_s)[1], f"{d.duration_s:.0f} s", d.brand or UNKNOWN_BRAND, d.category,\n             d.review or "") for k, d in enumerate(o.detections, 1)]\n    table = html_table(["STT", "Giờ", "Dài", "Nhãn hàng / spot", "Loại", "Cần nghe lại", "Nghe"], rows)\n    # the players are raw HTML: splice one into the end of each data row\n    parts = table.split("</tr>")\n    for k, clip in enumerate(o.clips, 1):\n        parts[k] += f\'<td>{clip}</td>\'\n    warn = "".join(html_note(f"⚠️ AI chưa phân tích được đoạn {hms(a)}–{hms(b)}: quảng cáo trong đoạn này có thể bị"\n                             " bỏ sót.") for a, b in o.not_analysed)\n    if o.learned:\n        warn += html_note(f"🧠 Đã lưu {len(o.learned)} quảng cáo mới làm spot trong 1_ThuVienQC/{PENDING_SUBDIR}: lần"\n                          " sau notebook dò chúng bằng vân tay. Nghe từng file: đúng thì đổi tên theo nhãn hàng rồi kéo"\n                          " ra 1_ThuVienQC, sai thì xoá.")\n    if o.asr_error:\n        warn += html_note(f"⚠️ Không chép lời được ({o.asr_error}) nên AI chưa tự tìm quảng cáo; kết quả chỉ gồm các"\n                          " spot bạn đưa vào. Hãy chụp màn hình lỗi này gửi người phụ trách.")\n    if o.lost:\n        minutes = sum(b - a for a, b in o.lost) / 60\n        warn += html_note(f"ℹ️ {len(o.lost)} đoạn (tổng {minutes:.0f} phút) không chép được lời, thường là bài hát."\n                          " Các đoạn này có dòng ⚠️ trong sheet Chép lời nếu cần nghe lại.")\n    return head + warn + "</tr>".join(parts)\n\n\n# --- notebook glue: kept here so it is tested locally with a fake Colab ---\n\ndef has_gpu() -> bool:\n    return shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0\n\n\ndef install(gpu: bool) -> None:\n    packages = ["av"] + (["faster-whisper"] if gpu else [])\n    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)\n\n\ndef default_asr() -> ASR:\n    from .transcript import transcribe\n    return lambda audio, progress: transcribe(audio, "cuda", progress)\n\n\ndef default_llm() -> LLM | None:\n    try:\n        from .classify import colab_gemini\n        return colab_gemini()\n    except ImportError:\n        return None\n', 'radioqc/transcript.py': '"""Speech-to-text for mode B.\n\nOne continuous pass over the whole recording with VAD, then a repair pass: Whisper sometimes\nreplaces 30-100 s of real speech over a music bed with a YouTube outro ("Hãy subscribe cho\nkênh…"), hiding whole ads. Those stretches are re-transcribed through differently placed windows\n(spec §6); what stays unreadable is reported as lost, never dropped silently. The same settings\nrun on Colab\'s GPU and on a local CPU.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nASR_MODEL = "large-v3-turbo"\nASR_SR = 16000\nASR_ARGS = {"language": "vi", "beam_size": 5, "vad_filter": True, "condition_on_previous_text": False}\n# Context on each side when re-transcribing a hallucinated stretch. What helps is not the meaning of the\n# context but where Whisper\'s 30 s windows fall on the music bed, and that is close to a coin toss: on\n# 25/9 one 102 s stretch was fully read with 30 or 45 s of context and not at all with 15 or 60 s; each\n# try read a stubborn stretch 30-50 % of the time (spec §6). Tries stop as soon as nothing is missing.\nRETRY_CONTEXTS_S = (60.0, 30.0, 10.0)\nRETRY_PIECE_S = 120.0    # longer stretches (songs, long breaks) are retried in pieces\nMERGE_GAP_S = 5.0        # hallucinated segments this close form one stretch\nLOST_MIN_S = 10.0        # an unrecovered gap at least this long is reported as lost\nFIRST_PASS_SHARE = 0.9   # of the progress bar; the repair pass fills the rest\n\n# Phrases Whisper invents over music or silence in Vietnamese (seen on the 25/9 broadcast).\n_HALLUCINATION = re.compile(\n    r"subscribe|đăng ký kênh|ghiền mì gõ|la la school|không bỏ lỡ những video|"\n    r"cảm ơn các bạn đã (theo dõi|xem)|like và share|bấm chuông", re.IGNORECASE)\n\n\n@dataclass(frozen=True)\nclass Line:\n    start: float   # seconds from the start of the file\n    end: float\n    text: str\n\n\ndef is_hallucination(text: str) -> bool:\n    return not text.strip() or bool(_HALLUCINATION.search(text))\n\n\ndef _merge(spans: list[tuple[float, float]], gap: float) -> list[tuple[float, float]]:\n    merged: list[list[float]] = []\n    for a, b in sorted(spans):\n        if merged and a - merged[-1][1] <= gap:\n            merged[-1][1] = max(merged[-1][1], b)\n        else:\n            merged.append([a, b])\n    return [(a, b) for a, b in merged]\n\n\ndef _gaps(a: float, b: float, covered: list[Line]) -> list[tuple[float, float]]:\n    """Parts of [a, b] not covered by `covered`, at least LOST_MIN_S long."""\n    out, pos = [], a\n    for ln in sorted(covered, key=lambda ln: ln.start):\n        if ln.start - pos >= LOST_MIN_S:\n            out.append((round(pos, 2), round(ln.start, 2)))\n        pos = max(pos, ln.end)\n    if b - pos >= LOST_MIN_S:\n        out.append((round(pos, 2), round(b, 2)))\n    return out\n\n\ndef _retry(model, audio: np.ndarray, p0: float, p1: float, context: float) -> list[Line]:\n    """Real lines whose midpoint falls in [p0, p1), from re-transcribing it with `context` s on each side."""\n    w0, w1 = max(0.0, p0 - context), min(audio.size / ASR_SR, p1 + context)\n    segs, _ = model.transcribe(audio[int(w0 * ASR_SR):int(w1 * ASR_SR)], **ASR_ARGS)\n    out = []\n    for s in segs:\n        s0, s1, text = w0 + s.start, w0 + s.end, s.text.strip()\n        if p0 <= (s0 + s1) / 2 < p1 and not is_hallucination(text):\n            out.append(Line(round(s0, 2), round(s1, 2), text))\n    return out\n\n\ndef repair(model, audio: np.ndarray, segments, progress: Callable[[float], None] | None = None\n           ) -> tuple[list[Line], list[tuple[float, float]]]:\n    """Keep real lines, re-transcribe hallucinated stretches and splice the speech back.\n\n    Each stretch is tried with every window in RETRY_CONTEXTS_S until nothing is missing; the attempt that\n    reads the most speech wins. Returns (lines, lost): `lost` are stretches still unreadable after that.\n    """\n    lines, bad = [], []\n    for s in segments:\n        text = s.text.strip()\n        if is_hallucination(text):\n            bad.append((float(s.start), float(s.end)))\n        else:\n            lines.append(Line(round(s.start, 2), round(s.end, 2), text))\n    lost: list[tuple[float, float]] = []\n    stretches = _merge(bad, MERGE_GAP_S)\n    total, done = sum(b - a for a, b in stretches) or 1.0, 0.0\n    for a, b in stretches:\n        recovered = []\n        for p0 in np.arange(a, b, RETRY_PIECE_S):\n            p1 = min(b, p0 + RETRY_PIECE_S)\n            best: list[Line] = []\n            for context in RETRY_CONTEXTS_S:\n                got = _retry(model, audio, p0, p1, context)\n                if sum(ln.end - ln.start for ln in got) > sum(ln.end - ln.start for ln in best):\n                    best = got\n                if not _gaps(p0, p1, best):\n                    break\n            recovered += best\n            done += p1 - p0\n            if progress:\n                progress(FIRST_PASS_SHARE + (1 - FIRST_PASS_SHARE) * done / total)\n        lines += recovered\n        lost += _gaps(a, b, recovered)\n    return sorted(lines, key=lambda ln: ln.start), lost\n\n\ndef transcribe(audio: np.ndarray, device: str = "cuda", progress: Callable[[float], None] | None = None,\n               model_name: str = ASR_MODEL) -> tuple[list[Line], list[tuple[float, float]]]:\n    """(lines, lost stretches) for 16 kHz mono audio. Needs the `faster-whisper` package."""\n    from faster_whisper import WhisperModel  # heavy, and only mode B needs it\n    try:  # the model is public: "unauthenticated requests to the HF Hub" is noise that scares users\n        from huggingface_hub.utils import logging as hf_logging\n        hf_logging.set_verbosity_error()\n    except ImportError:\n        pass\n\n    model = WhisperModel(model_name, device=device, compute_type="float16" if device == "cuda" else "int8")\n    segments, _ = model.transcribe(audio, **ASR_ARGS)\n    duration = max(audio.size / ASR_SR, 1e-6)\n    raw = []\n    for s in segments:\n        if progress:\n            progress(FIRST_PASS_SHARE * min(1.0, s.end / duration))\n        raw.append(s)\n    lines, lost = repair(model, audio, raw, progress)\n    if progress:\n        progress(1.0)\n    return lines, lost\n', 'radioqc/ui.py': '"""Tiny HTML helpers for the notebook, so its cells need neither pandas nor anything else."""\nfrom __future__ import annotations\n\nimport html\nfrom collections.abc import Iterable, Sequence\n\n_ALERT = {"LỖI": "color:#c00;font-weight:bold", "CẢNH BÁO": "color:#b60", "Cần nghe lại": "color:#b60",\n          "CHỜ FILE": "color:#1a73e8"}\n\n\ndef _style(value: str) -> str:\n    return next((css for prefix, css in _ALERT.items() if value.startswith(prefix)), "")\n\n\ndef html_table(headers: Sequence[str], rows: Iterable[Sequence[object]]) -> str:\n    """An escaped HTML table; error, warning and waiting cells are coloured."""\n    def cell(tag: str, value: object) -> str:\n        text = str(value)\n        return f\'<{tag} style="text-align:left;{_style(text)}">{html.escape(text)}</{tag}>\'\n\n    head = "<tr>" + "".join(cell("th", h) for h in headers) + "</tr>"\n    body = "".join("<tr>" + "".join(cell("td", v) for v in row) + "</tr>" for row in rows)\n    return f"<table>{head}{body}</table>"\n\n\ndef html_note(text: str) -> str:\n    """A highlighted \'what to do next\' box, readable in light and dark themes."""\n    return (\'<div style="margin:8px 0;padding:10px 14px;border-left:4px solid #1a73e8;\'\n            f\'background:#e8f0fe;color:#174ea6;font-size:14px">{html.escape(text)}</div>\')\n'}
_base = pathlib.Path("/content/radioqc_code")
for _rel, _src in _FILES.items():
    _p = _base / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_src, encoding="utf-8")
if str(_base) not in sys.path:
    sys.path.insert(0, str(_base))
for _m in [m for m in sys.modules if m == "radioqc" or m.startswith("radioqc.")]:
    del sys.modules[_m]  # pick up new code when this cell is re-run
from google.colab import files
from IPython.display import HTML, display
from radioqc import simple
from radioqc.pipeline import save_uploads
from radioqc.ui import html_note
_hop = None
if NGUON_FILE == "Google Drive":
    from google.colab import drive
    CHON, KET_QUA = [], None
    try:
        drive.mount("/content/drive")
    except Exception as _e:  # the user closed or refused the Drive permission window
        display(HTML(html_note(f"Chưa vào được Google Drive ({_e}). Bấm Runtime → Run all lại và chọn Cho phép, "
                               "hoặc ở ô ⚙️ Tuỳ chọn chọn NGUON_FILE = Tải lên từ máy.")))
    else:
        _hop = simple.drive_inbox(simple.DRIVE_ROOT, THU_MUC_DRIVE, redo=CHAY_LAI_TAT_CA)
        display(HTML(html_note(_hop.message)))
        CHON, KET_QUA = _hop.broadcasts, _hop.result_dir
else:
    print("Chọn file phát sóng (có file spot quảng cáo thì chọn kèm):")
    try:
        _uploaded = files.upload()
    except Exception:  # the user cancelled the file picker
        _uploaded = {}
    CHON, _bo_qua = save_uploads(_uploaded, simple.UPLOAD_DIR)
    KET_QUA = simple.RESULT_DIR
    for _name in _bo_qua:
        print(f"⚠️ Bỏ qua {_name}: không phải file âm thanh")
    if not CHON:
        display(HTML(html_note("Chưa chọn file nào. Bấm Runtime → Run all lần nữa rồi chọn file.")))


In [ ]:
#@title 2️⃣ Tìm quảng cáo và tải kết quả về máy
from radioqc import simple
if CHON:
    _gpu = simple.has_gpu()
    print("Đang chuẩn bị…" + (" (có GPU: AI sẽ tự nhận diện quảng cáo)" if _gpu else ""))
    simple.install(_gpu)
    _kho = None
    if _hop is not None and _hop.spots:
        from radioqc.library import build_library
        _kho = build_library(_hop.spot_dir, pathlib.Path("/content/kho_spot_cache"), log=lambda m: None)
    _hoc = _hop.spot_dir / simple.PENDING_SUBDIR if _hop is not None and TU_HOC_SPOT else None
    _res = simple.run(CHON, KET_QUA, extra_library=_kho, learn_dir=_hoc,
                      asr=simple.default_asr() if _gpu else None, llm=simple.default_llm() if _gpu else None)
    for _o in _res:
        display(HTML(simple.results_html(_o)))
        if _o.report:
            files.download(str(_o.report))
    if any(_o.report for _o in _res):
        _ban_sao = f" và có một bản trong Google Drive: {THU_MUC_DRIVE}/3_KetQua" if _hop is not None else ""
        display(HTML(html_note(f"✅ Xong. File Excel kết quả đã tải về máy (thư mục Tải xuống / Downloads){_ban_sao}.")))
